# Hierarchical ILD-XR: Binary Screening + 5-Class Pathology Mapping

## Purpose
Improve publishability by reframing the task as a two-level hierarchy:

| Level | Task | Classes | Expected F1 | Clinical Role |
|-------|------|---------|-------------|---------------|
| **Primary** | Normal vs Any ILD | 2 | **>85%** | Triage screening |
| **Secondary** | 5-class pathology | 5 | ~0.45-0.55 weighted | Treatment planning + biomarkers |

## Key design changes from NB02
- **All 113 patients** in repeated 5-fold GroupKFold (drop the damaging 11-patient held-out)
- **Shared 3D ResNet-18 encoder** (MedicalNet pretrained) with two classification heads
- **Binary head** trained on ALL patches (much more balanced, ~50/50)
- **5-class pathology head** trained only on ILD-positive patches
- **Bootstrap 95% CIs** on all metrics (1000 iterations)
- **Calibration**: ECE, Brier, reliability diagrams on real Softmax probabilities
- **Statistical significance**: McNemar's test / paired bootstrap against baselines
- **Repeated 5-fold**: 3 repeats with different patient permutations

## Full-volume cascade (proposal)
`CT -> lungmask -> sliding-window patch Softmax -> 3D pathology map -> classification + biomarkers`
(not literature U-Net segmenter then separate classifier).

## Outputs
- `hierarchical_cv_results.json` — per-fold metrics for both heads
- `hierarchical_bootstrap_metrics.json` — metrics with 95% CI
- `hierarchical_calibration.json` — ECE / Brier with CI
- `hierarchical_roc.json` — per-class AUC with CI
- `hierarchical_ablation.json` — controlled ablations
- `hierarchical_cascade_summary.json` / `hierarchical_cascade_patients.csv` / `cascade_maps/`
- `hierarchical_biomarker.csv` — per-patient volumetric biomarkers
- Figures: `roc_curves_hierarchical.png`, `reliability_hierarchical.png`, `confusion_hierarchical.png`

## Hardware
Limited VRAM: use batch size 2 with gradient accumulation; 3D patches (16, 64, 64).


## 1 — Setup & Configuration

In [ ]:
import os, json, gc, random, warnings, itertools, math
from pathlib import Path
from dataclasses import dataclass
from copy import deepcopy
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pydicom
import urllib.request
import joblib

from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.metrics import (
    f1_score, accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    brier_score_loss, matthews_corrcoef, cohen_kappa_score,
)
from sklearn.calibration import calibration_curve
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from scipy.stats import spearmanr, pearsonr, chi2_contingency

warnings.filterwarnings('ignore')

# ── Load local.env ──
def _load_local_env(path):
    if not path.is_file(): return
    for line in path.read_text(encoding='utf-8').splitlines():
        line = line.strip()
        if not line or line.startswith('#') or '=' not in line: continue
        key, _, value = line.partition('=')
        os.environ.setdefault(key.strip(), value.strip())

for p in [Path.cwd() / 'local.env',
         Path.cwd() / 'Experimentations' / 'local.env',
         Path.cwd() / 'Local' / 'local.env',
         Path('Local/local.env'),
         Path('Experimentations/local.env')]:
    _load_local_env(p)

# ── Hardware ──
IS_KAGGLE = os.path.isdir('/kaggle/input')
IS_LOCAL = not IS_KAGGLE
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
HARDWARE = os.environ.get('ILD_HARDWARE', '').strip()

# ── Reproducibility ──
GLOBAL_SEED = 42
random.seed(GLOBAL_SEED)
np.random.seed(GLOBAL_SEED)
torch.manual_seed(GLOBAL_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(GLOBAL_SEED)

# ── Paths ──
_DEFAULT_MEDGIFT = ''
MEDGIFT_ROOT = os.environ.get('ILD_MEDGIFT_ROOT', _DEFAULT_MEDGIFT)
HOME_DIR = '/kaggle/working/' if IS_KAGGLE else 'Results'
LUNG_CKPT_DIR = os.environ.get('ILD_MODELS_DIR', os.path.join(HOME_DIR, 'models')).strip() or os.path.join(HOME_DIR, 'models')
MODELS_DIR = LUNG_CKPT_DIR
EXPORTS_DIR = os.environ.get('ILD_EXPORTS_DIR', os.path.join(HOME_DIR, 'exports_3d_seg'))
FIGURES_DIR = os.environ.get('ILD_FIGURES_DIR', os.path.join('exports', 'figures'))

def find_lung_mask_base(data_dir):
    for p in [os.path.join(data_dir, 'lungMasks')]:
        if os.path.isdir(p): return p
    parent = os.path.dirname(os.path.abspath(data_dir))
    p = os.path.join(parent, 'lungMasks')
    if os.path.isdir(p): return p
    return None

LUNG_MASK_BASE = os.environ.get('ILD_LUNG_MASK_BASE', '').strip() or find_lung_mask_base(MEDGIFT_ROOT)
REQUIRE_REAL_LUNG = os.environ.get('ILD_REQUIRE_REAL_LUNG', '1') == '1'

for d in (MODELS_DIR, EXPORTS_DIR, FIGURES_DIR):
    os.makedirs(d, exist_ok=True)

# ── Hyperparameters (from local.env with defaults) ──
def _parse_patch3(text, fallback):
    try:
        parts = [int(x.strip()) for x in text.split(',')]
        if len(parts) == 3 and all(p > 0 for p in parts): return tuple(parts)
    except ValueError: pass
    return fallback

QUICK_RUN = os.environ.get('QUICK_RUN', '0') == '1'
SMOKE_FOLD1 = os.environ.get('ILD_SMOKE_FOLD1', '0') == '1'
RUN_TRAIN = os.environ.get('ILD_RUN_TRAIN', '1') == '1'
PATCH_AUGMENT = os.environ.get('ILD_PATCH_AUGMENT', '1') == '1'

# Patch geometry
CLS_PATCH_SIZE = _parse_patch3(os.environ.get('ILD_CLS_PATCH_SIZE', ''), (16, 64, 64))
INPLANE = int(os.environ.get('ILD_INPLANE', '128'))
HU_CLIP = (-1350.0, 150.0)

# Training
ENCODER_EPOCHS = int(os.environ.get('ILD_ENCODER_EPOCHS', '30'))
PATCHES_PER_EPOCH = int(os.environ.get('ILD_PATCHES_PER_EPOCH', '2400'))
VAL_PATCHES = int(os.environ.get('ILD_VAL_PATCHES', '300'))
FEAT_BATCH_SIZE = int(os.environ.get('ILD_FEAT_BATCH_SIZE', '8'))
GRAD_ACCUM_STEPS = max(1, int(os.environ.get('ILD_GRAD_ACCUM_STEPS', '1')))
EFFECTIVE_BATCH = FEAT_BATCH_SIZE * GRAD_ACCUM_STEPS
NUM_WORKERS = int(os.environ.get('ILD_NUM_WORKERS', '0' if IS_LOCAL else '2'))
VOL_CACHE_PATIENTS = int(os.environ.get('ILD_VOLUME_CACHE_PATIENTS', '6' if IS_LOCAL else '2'))
HEAD_DROPOUT = float(os.environ.get('ILD_HEAD_DROPOUT', '0.4'))

# Discriminative fine-tuning (binary-primary; no probe phase by default)
# Probe→unfreeze was killing most folds.
WARMUP_EPOCHS = int(os.environ.get('ILD_WARMUP_EPOCHS', '0'))  # 0 = discriminative from epoch 1
PROBE_LR = float(os.environ.get('ILD_PROBE_LR', '1e-3'))  # unused when WARMUP_EPOCHS=0
FINETUNE_LR = float(os.environ.get('ILD_FINETUNE_LR', '1e-5'))
FINETUNE_HEAD_LR = float(os.environ.get('ILD_FINETUNE_HEAD_LR', '1e-4'))
FINETUNE_WD = float(os.environ.get('ILD_FINETUNE_WD', '1e-4'))
FINETUNE_PATIENCE = int(os.environ.get('ILD_FINETUNE_PATIENCE', '15'))
UNFREEZE_BLOCKS = tuple(b.strip() for b in os.environ.get('ILD_UNFREEZE_BLOCKS', 'layer3,layer4').split(',') if b.strip())
MIXUP_ALPHA = float(os.environ.get('ILD_MIXUP_ALPHA', '0.2'))
LABEL_SMOOTHING = float(os.environ.get('ILD_LABEL_SMOOTHING', '0.1'))
# Discriminative LRs (used when WARMUP_EPOCHS=0)
LR_STEM = float(os.environ.get('ILD_LR_STEM', '1e-6'))
LR_LAYER3 = float(os.environ.get('ILD_LR_LAYER3', '1e-5'))
LR_LAYER4 = float(os.environ.get('ILD_LR_LAYER4', '3e-5'))
LR_HEAD = float(os.environ.get('ILD_LR_HEAD', str(FINETUNE_HEAD_LR)))
BINARY_ONLY_TRAIN = os.environ.get('ILD_BINARY_ONLY_TRAIN', '1') == '1'

# Med3D pretrained weights
MED3D_WEIGHTS = os.environ.get('ILD_MED3D_WEIGHTS', '').strip()
if MED3D_WEIGHTS and not os.path.isabs(MED3D_WEIGHTS) and not os.path.isfile(MED3D_WEIGHTS):
    # Resolve relative weight path against MODELS_DIR / LUNG_CKPT_DIR
    alt = os.path.join(MODELS_DIR, os.path.basename(MED3D_WEIGHTS))
    if os.path.isfile(alt):
        MED3D_WEIGHTS = alt
MED3D_URL = os.environ.get(
    'ILD_MED3D_URL',
    'https://huggingface.co/TencentMedicalNet/MedicalNet-Resnet18/resolve/main/resnet_18.pth',
).strip()
REQUIRE_PRETRAIN = os.environ.get('ILD_REQUIRE_PRETRAIN', '1') == '1'

# Repeated CV
N_REPEATS = int(os.environ.get('ILD_N_REPEATS', '3' if QUICK_RUN else '5'))
N_FOLDS = int(os.environ.get('ILD_N_FOLDS', '2' if (QUICK_RUN or SMOKE_FOLD1) else '5'))
N_BOOT = int(os.environ.get('ILD_N_BOOT', '100' if QUICK_RUN else '1000'))

# Patch mining gates
MIN_PATHOLOGY_VOXELS = int(os.environ.get('ILD_MIN_PATHOLOGY_VOXELS', '80'))
MIN_TARGET_CLASS_FRAC = float(os.environ.get('ILD_MIN_TARGET_CLASS_FRAC', '0.15'))
MIN_PATCH_LUNG_FRAC = float(os.environ.get('ILD_MIN_PATCH_LUNG_FRAC_CLS', '0.20'))
MIN_NORMAL_LUNG_FRAC = float(os.environ.get('ILD_MIN_NORMAL_LUNG_FRAC', '0.50'))
MIN_PATCHES_PER_CLASS = int(os.environ.get('ILD_MIN_PATCHES_PER_CLASS', '60'))
PATHOLOGY_PATCH_QUOTA = float(os.environ.get('ILD_PATHOLOGY_PATCH_QUOTA', '0.85'))
MAX_CLASS_ORIGIN_POOL = int(os.environ.get('ILD_MAX_CLASS_ORIGIN_POOL', '1024'))
PATCH_LABEL_MODE = os.environ.get('ILD_PATCH_LABEL_MODE', 'dominant').strip().lower()

# Full-volume cascade inference
INFER_DENSE_STRIDE = _parse_patch3(os.environ.get('ILD_INFER_DENSE_STRIDE', ''), (4, 8, 8))
INFER_MAX_PATCHES = int(os.environ.get('ILD_INFER_MAX_PATCHES', '8000' if not QUICK_RUN else '400'))
INFER_CLEANUP_EVERY = int(os.environ.get('ILD_INFER_CLEANUP_EVERY', '64'))
RUN_CASCADE = os.environ.get('ILD_RUN_CASCADE', '1') == '1'
CASCADE_MAX_PATIENTS = int(os.environ.get('ILD_CASCADE_MAX_PATIENTS', '0'))  # 0 = all
CASCADE_SAVE_MAPS = os.environ.get('ILD_CASCADE_SAVE_MAPS', '1') == '1'

# ── Class mappings ──
# Original MedGIFT classes
ORIGINAL_CLASS_NAMES = ['Normal', 'Emphysema', 'Fibrosis', 'Ground Glass', 'Micronodules', 'Consolidation']

# Hierarchical: Normal / Fibrotic / Non-fibrotic
# Fibrotic = Fibrosis (2) + Consolidation (5)
# Non-fibrotic = Emphysema (1) + Ground Glass (3) + Micronodules (4)
HIERARCHY_MAP = {0: 0, 1: 2, 2: 1, 3: 2, 4: 2, 5: 1}  # orig -> hier
HIERARCHY_CLASSES = ['Normal', 'Fibrotic', 'NonFibrotic']
N_HIER_CLASSES = 3

# Binary: Normal (0) vs ILD (1-5)
BINARY_CLASSES = ['Normal', 'ILD']
N_BINARY_CLASSES = 2

# 6-class pathology (secondary, only ILD patches)
PATHOLOGY_CLASSES = ORIGINAL_CLASS_NAMES[1:]  # ['Emphysema', 'Fibrosis', 'Ground Glass', 'Micronodules', 'Consolidation']
N_PATH_CLASSES = 5

SEG_NUM_CLASSES = 6  # original MedGIFT labels
MEDGIFT_TO_CLS_FALLBACK = {1:0, 2:1, 3:3, 4:2, 5:4, 6:5, 8:5, 11:4, 14:2}

print(f'Device: {device}')
if HARDWARE: print(f'HARDWARE profile: {HARDWARE}')
print(f'MEDGIFT_ROOT: {MEDGIFT_ROOT} (exists={os.path.isdir(MEDGIFT_ROOT)})')
print(f'LUNG_MASK_BASE: {LUNG_MASK_BASE} (exists={bool(LUNG_MASK_BASE and os.path.isdir(LUNG_MASK_BASE))})')
print(f'MODELS_DIR/LUNG_CKPT_DIR: {MODELS_DIR}')
print(f'MED3D_WEIGHTS: {MED3D_WEIGHTS} (exists={bool(MED3D_WEIGHTS and os.path.isfile(MED3D_WEIGHTS))})')
print(f'RUN_TRAIN={RUN_TRAIN} REQUIRE_PRETRAIN={REQUIRE_PRETRAIN} PATCH_AUGMENT={PATCH_AUGMENT}')
print(f'CLS_PATCH={CLS_PATCH_SIZE} PATCHES/epoch={PATCHES_PER_EPOCH} BATCH={FEAT_BATCH_SIZE}x{GRAD_ACCUM_STEPS} (eff={EFFECTIVE_BATCH})')
print(f'Repeated CV: {N_REPEATS}x{N_FOLDS}-fold | Bootstrap: {N_BOOT} iters')
print(f'MixUp alpha={MIXUP_ALPHA} | Label smoothing={LABEL_SMOOTHING}')
print(f'Train schedule: WARMUP={WARMUP_EPOCHS} BINARY_ONLY={BINARY_ONLY_TRAIN} PATIENCE={FINETUNE_PATIENCE}')
print(f'Disc LR: stem/l1/l2={LR_STEM} layer3={LR_LAYER3} layer4={LR_LAYER4} head={LR_HEAD}')
print(f'Hierarchy: {HIERARCHY_CLASSES}')
print(f'Binary: {BINARY_CLASSES}')
print(f'5-class pathology: {PATHOLOGY_CLASSES}')
print(f'Cascade: RUN={RUN_CASCADE} stride={INFER_DENSE_STRIDE} max_patches={INFER_MAX_PATCHES}')


## 2 — Data Loading & Patient Enumeration

In [ ]:
# ── DICOM I/O (identical to NB01/NB02) ──
def dicom_to_hu(ds):
    arr = ds.pixel_array.astype(np.float32)
    slope = float(getattr(ds, 'RescaleSlope', 1.0) or 1.0)
    intercept = float(getattr(ds, 'RescaleIntercept', 0.0) or 0.0)
    return arr * slope + intercept

def load_dicom_volume(folder_path, as_hu=False):
    if not os.path.isdir(folder_path): return None
    files = [f for f in os.listdir(folder_path) if f.lower().endswith('.dcm')]
    if not files: return None
    datasets = [pydicom.dcmread(os.path.join(folder_path, name)) for name in files]
    datasets.sort(key=lambda ds: float(getattr(ds, 'SliceLocation', getattr(ds, 'InstanceNumber', 0))))
    slices = [dicom_to_hu(ds) if as_hu else ds.pixel_array.astype(np.float32) for ds in datasets]
    return np.stack(slices, axis=-1)

def find_roi_path(data_dir):
    if data_dir and os.path.exists(data_dir): return data_dir
    return None

def _dcm_in(d):
    return os.path.isdir(d) and any(f.lower().endswith('.dcm') for f in os.listdir(d))

def _dir_has_ct(p):
    if _dcm_in(p): return True
    for name in ('volumeCT', 'ct', 'CT', 'VolumeCT'):
        if _dcm_in(os.path.join(p, name)): return True
    return False

def _dir_has_roi(p):
    for sub in ('roi_mask', 'ROI', 'roi'):
        if _dcm_in(os.path.join(p, sub)): return True
    return False

def _series_subdirs(p):
    out = []
    for s in sorted(os.listdir(p)):
        sp = os.path.join(p, s)
        if os.path.isdir(sp) and _dir_has_ct(sp) and _dir_has_roi(sp):
            out.append((s, sp))
    return out

def list_patient_units(roi_base):
    units = []
    if not os.path.isdir(roi_base): return units
    def add_patient(patient_dir, group, cohort):
        if _dir_has_ct(patient_dir) and _dir_has_roi(patient_dir):
            units.append({'load_path': patient_dir, 'uid': group, 'group': group,
                          'cohort': cohort, 'series': None, 'missing': False})
            return
        subs = _series_subdirs(patient_dir)
        if subs:
            for sname, spath in subs:
                units.append({'load_path': spath, 'uid': f'{group}__{sname}', 'group': group,
                              'cohort': cohort, 'series': sname, 'missing': False})
            return
        units.append({'load_path': patient_dir, 'uid': group, 'group': group,
                      'cohort': cohort, 'series': None, 'missing': True})
    for name in sorted(os.listdir(roi_base)):
        full = os.path.join(roi_base, name)
        if not os.path.isdir(full): continue
        if name == 'HRCT_pilot':
            for sub in sorted(os.listdir(full)):
                sub_full = os.path.join(full, sub)
                if os.path.isdir(sub_full): add_patient(sub_full, sub, 'pilot')
        else:
            add_patient(full, name, 'main')
    return units

def load_patient_ct(patient_path):
    for name in ['volumeCT', 'ct', 'CT', 'VolumeCT']:
        vol = load_dicom_volume(os.path.join(patient_path, name), as_hu=True)
        if vol is not None: return vol
    return load_dicom_volume(patient_path, as_hu=True)

def load_roi_volume(patient_path):
    for sub in ['roi_mask', 'ROI', 'roi']:
        vol = load_dicom_volume(os.path.join(patient_path, sub), as_hu=False)
        if vol is not None: return vol
    return None

def normalize_hu_volume(vol_hu, clip=HU_CLIP):
    lo, hi = clip
    v = np.clip(vol_hu.astype(np.float32), lo, hi)
    return ((v - lo) / (hi - lo + 1e-8)).astype(np.float32)

def resize_volume_inplane(vol, size, mode='bilinear'):
    h, w, d = vol.shape
    if h == size and w == size: return vol.astype(np.float32)
    out = np.zeros((size, size, d), dtype=np.float32)
    for z in range(d):
        t = torch.from_numpy(vol[:,:,z].astype(np.float32)).unsqueeze(0).unsqueeze(0)
        kwargs = {'size': (size, size), 'mode': mode}
        if mode in ('bilinear', 'bicubic'): kwargs['align_corners'] = False
        out[:,:,z] = F.interpolate(t, **kwargs).squeeze().numpy()
    return out

def roi_volume_to_seg_labels(roi_vol, lung_mask):
    labels = np.zeros(roi_vol.shape, dtype=np.uint8)
    lung = lung_mask > 0.5
    labels[lung] = 0
    rounded = np.rint(roi_vol).astype(np.int32)
    for cls_idx in range(1, 6):
        labels[lung & (rounded == cls_idx)] = cls_idx
    for raw_id, canonical in MEDGIFT_TO_CLS_FALLBACK.items():
        cls_idx = int(canonical)
        if cls_idx == 0: continue
        labels[lung & (rounded == int(raw_id))] = cls_idx
    return labels

def load_patient_volumes_3d(patient_roi_path, data_dir, inplane=128, pid_override=None):
    roi_base = find_roi_path(data_dir)
    ct_base = roi_base
    lung_base = LUNG_MASK_BASE
    pid = pid_override if pid_override is not None else os.path.basename(patient_roi_path.rstrip('/\\'))
    roi_vol = load_roi_volume(patient_roi_path)
    ct_vol = load_patient_ct(patient_roi_path)
    if ct_vol is None: return None
    if roi_vol is None: roi_vol = np.zeros_like(ct_vol)
    lung_vol = None
    if lung_base:
        npy_path = os.path.join(lung_base, str(pid) + '.npy')
        if os.path.isfile(npy_path):
            lung_vol = np.load(npy_path)
    if lung_vol is None:
        if REQUIRE_REAL_LUNG:
            raise FileNotFoundError(f'No cached lung mask for {pid}. Run 01_segmentation.ipynb first.')
        ct_hu = ct_vol if ct_vol.max() > 2 else ct_vol * (HU_CLIP[1]-HU_CLIP[0]) + HU_CLIP[0]
        lung_vol = np.stack([((ct_hu[:,:,z] >= -1000) & (ct_hu[:,:,z] <= -200)).astype(np.float32) for z in range(ct_hu.shape[2])], axis=-1)
    min_z = min(ct_vol.shape[2], roi_vol.shape[2], lung_vol.shape[2])
    ct_vol, roi_vol, lung_vol = ct_vol[:,:,:min_z], roi_vol[:,:,:min_z], lung_vol[:,:,:min_z]
    ct_norm = normalize_hu_volume(ct_vol)
    lung_binary = (lung_vol > 0).astype(np.float32)
    ct_norm = resize_volume_inplane(ct_norm, inplane, mode='bilinear')
    lung_binary = resize_volume_inplane(lung_binary, inplane, mode='nearest')
    roi_vol = resize_volume_inplane(roi_vol.astype(np.float32), inplane, mode='nearest')
    seg_labels = roi_volume_to_seg_labels(roi_vol, lung_binary)
    ct_norm = np.transpose(ct_norm, (2, 0, 1))
    lung_binary = np.transpose(lung_binary, (2, 0, 1))
    seg_labels = np.transpose(seg_labels, (2, 0, 1))
    return {'pid': str(pid), 'ct_norm': ct_norm.astype(np.float32),
            'lung_mask': lung_binary, 'seg_labels': seg_labels, 'shape': ct_norm.shape}

# ── Build patient index ──
print('Building patient volume index...')
if not os.path.isdir(MEDGIFT_ROOT):
    raise FileNotFoundError(
        f'MEDGIFT_ROOT does not exist: {MEDGIFT_ROOT}. '
        'Set ILD_MEDGIFT_ROOT (environment variable or local.env).'
    )
roi_base = find_roi_path(MEDGIFT_ROOT)
if not roi_base or not os.path.isdir(roi_base):
    raise FileNotFoundError(
        f'No ROI volume root under MEDGIFT_ROOT={MEDGIFT_ROOT}. '
        'Set ILD_MEDGIFT_ROOT to your MedGIFT volume root.'
    )
units = list_patient_units(roi_base)

patient_records = []
dropped = []
loaded_groups = set()
for u in tqdm(units, desc='Indexing'):
    if u.get('missing'):
        dropped.append((u['uid'], 'no CT/ROI found')); continue
    try:
        packed = load_patient_volumes_3d(u['load_path'], MEDGIFT_ROOT, inplane=INPLANE, pid_override=u['uid'])
    except FileNotFoundError as e:
        dropped.append((u['uid'], f'missing_lung_mask: {e}')); continue
    if packed is None:
        dropped.append((u['uid'], 'load failed')); continue
    lung_frac = float(packed['lung_mask'].mean())
    if lung_frac < 0.02:
        dropped.append((u['uid'], f'lung_frac={lung_frac:.3f}')); continue
    lung = packed['lung_mask'] > 0.5
    in_lung = packed['seg_labels'][lung]
    class_counts = np.bincount(in_lung.astype(np.int64), minlength=SEG_NUM_CLASSES)
    # Hierarchical labels
    hier_counts = np.zeros(N_HIER_CLASSES, dtype=np.int64)
    for orig_c in range(SEG_NUM_CLASSES):
        hier_c = HIERARCHY_MAP[orig_c]
        hier_counts[hier_c] += class_counts[orig_c]
    has_ild = int((packed['seg_labels'] > 0).any())
    patient_records.append({
        'pid': packed['pid'], 'group': u['group'], 'cohort': u['cohort'],
        'path': u['load_path'],
        'depth': int(packed['ct_norm'].shape[0]),
        'class_counts': class_counts.tolist(),
        'hier_counts': hier_counts.tolist(),
        'has_ild': has_ild,
        'lung_frac': lung_frac,
    })
    loaded_groups.add(u['group'])
    del packed

all_groups = sorted(loaded_groups)
n_ild = sum(1 for r in patient_records if r['has_ild'])
n_normal = sum(1 for r in patient_records if not r['has_ild'])
print(f'Indexed {len(patient_records)} series | {len(all_groups)} patients | ILD+={n_ild} Normal={n_normal}')
if dropped:
    print(f'Dropped {len(dropped)} units')
    for uid, why in dropped[:5]:
        print(f'  - {uid}: {why}')


## 2b — Lung-Mask Cache Audit (113 patients)


In [ ]:
# Verify CT + ROI + real lung mask availability for every indexed series.
# Loader uses flat `{pid}.npy` under LUNG_MASK_BASE (same as NB01/NB02 cache).
# Series without real masks are dropped when REQUIRE_REAL_LUNG=1 (no HU fallback).

def _patient_has_lung_mask(pid, load_path, lung_base):
    if not lung_base or not os.path.isdir(lung_base):
        return False, 'lung_base_missing'
    pid = str(pid)
    group = pid.split('__')[0] if '__' in pid else pid
    base_name = os.path.basename(str(load_path).rstrip('/\\'))
    # Match load_patient_volumes_3d: prefer flat .npy caches
    file_candidates = [
        os.path.join(lung_base, pid + '.npy'),
        os.path.join(lung_base, group + '.npy'),
        os.path.join(lung_base, base_name + '.npy'),
        os.path.join(lung_base, pid + '.npz'),
        os.path.join(lung_base, group + '.npz'),
    ]
    for fp in file_candidates:
        if os.path.isfile(fp):
            return True, fp
    dir_candidates = [
        os.path.join(lung_base, pid),
        os.path.join(lung_base, group),
        os.path.join(lung_base, base_name),
    ]
    for c in dir_candidates:
        if not os.path.isdir(c):
            continue
        for root, _dirs, files in os.walk(c):
            for fn in files:
                low = fn.lower()
                if low.endswith(('.nii', '.nii.gz', '.npy', '.npz', '.dcm')):
                    return True, os.path.join(root, fn)
                if 'mask' in low or 'lung' in low:
                    return True, os.path.join(root, fn)
    return False, 'no_mask_file'


audit_rows = []
missing_masks = []
for rec in patient_records:
    ok, detail = _patient_has_lung_mask(rec['pid'], rec['path'], LUNG_MASK_BASE)
    audit_rows.append({
        'pid': rec['pid'], 'group': rec['group'], 'cohort': rec.get('cohort'),
        'has_ild': int(rec.get('has_ild', 0)), 'mask_ok': int(ok), 'detail': detail,
    })
    if not ok:
        missing_masks.append((rec['pid'], detail))

audit_df = pd.DataFrame(audit_rows)
n_ok = int(audit_df['mask_ok'].sum()) if len(audit_df) else 0
n_tot = len(audit_df)
n_groups = audit_df['group'].nunique() if len(audit_df) else 0
print(f'Mask audit: {n_ok}/{n_tot} series with lung masks | {n_groups} unique patients')
print(f'LUNG_MASK_BASE={LUNG_MASK_BASE} exists={bool(LUNG_MASK_BASE and os.path.isdir(LUNG_MASK_BASE))}')
print(f'MEDGIFT_ROOT={MEDGIFT_ROOT} exists={os.path.isdir(MEDGIFT_ROOT)}')

audit_path = os.path.join(EXPORTS_DIR, 'hierarchical_mask_audit.csv')
audit_df.to_csv(audit_path, index=False)
print(f'Saved: {audit_path}')

if n_tot == 0:
    raise RuntimeError(
        f'No patients indexed from MEDGIFT_ROOT={MEDGIFT_ROOT}. '
        'Confirm ILD_MEDGIFT_ROOT points at your MedGIFT volume root.'
    )

if missing_masks:
    preview = ', '.join(f'{p}({why})' for p, why in missing_masks[:8])
    print(f'WARNING: {len(missing_masks)}/{n_tot} series lack lung-mask files. Examples: {preview}')
    if REQUIRE_REAL_LUNG:
        keep = {r['pid'] for r in audit_rows if r['mask_ok']}
        before = len(patient_records)
        patient_records = [r for r in patient_records if r['pid'] in keep]
        all_groups = sorted({r['group'] for r in patient_records})
        n_ild = sum(1 for r in patient_records if r['has_ild'])
        n_normal = sum(1 for r in patient_records if not r['has_ild'])
        print(
            f'Dropped {before - len(patient_records)} series without real masks '
            f'(ILD_REQUIRE_REAL_LUNG=1). Kept {len(patient_records)} series / '
            f'{len(all_groups)} patients | ILD+={n_ild} Normal={n_normal}'
        )
        if len(patient_records) == 0:
            raise RuntimeError('No series left after dropping those without lung masks.')
    else:
        print('ILD_REQUIRE_REAL_LUNG=0 — keeping series; HU-threshold fallback may apply at load.')

print(
    f"Mask audit PASS ({len({r['group'] for r in patient_records})} patients / "
    f'{len(patient_records)} series retained).'
)


## 3 — Stratified Patch Mining

Two patch types are mined:
- **Binary patches**: all lung patches, labeled Normal (0) vs ILD (1) — used for primary classifier
- **Pathology patches**: only ILD-positive patches, labeled 1-of-5 — used for secondary classifier

In [ ]:
@dataclass
class PatchRecord:
    pid: str
    origin: tuple
    label: int        # Original 6-class label (0-5)
    hier_label: int   # 3-class hierarchical label (0-2)
    binary_label: int # Binary: 0=Normal, 1=ILD
    group: str = ''

def extract_patch(volume, origin, patch_size):
    oz, oy, ox = origin
    pd, ph, pw = patch_size
    patch = volume[oz:min(volume.shape[0], oz+pd),
                   oy:min(volume.shape[1], oy+ph),
                   ox:min(volume.shape[2], ox+pw)]
    if patch.shape != (pd, ph, pw):
        out = np.zeros((pd, ph, pw), dtype=volume.dtype)
        out[:patch.shape[0], :patch.shape[1], :patch.shape[2]] = patch
        return out
    return patch

def patch_dominant_label(seg_patch, lung_patch):
    lung = lung_patch > 0.5
    lung_n = int(lung.sum())
    if lung_n == 0: return None
    labels_in_lung = seg_patch[lung].astype(np.int64)
    counts = np.bincount(labels_in_lung, minlength=SEG_NUM_CLASSES)
    path_counts = counts[1:].copy()
    if path_counts.sum() == 0: return 0
    return int(np.argmax(path_counts) + 1)

def patch_label_ok(seg_patch, lung_patch, target_cls):
    lung = lung_patch > 0.5
    lung_n = float(lung.sum())
    if lung_n <= 0: return False
    lung_frac = lung_n / float(seg_patch.size)
    if lung_frac < MIN_PATCH_LUNG_FRAC: return False
    in_lung = seg_patch[lung]
    if target_cls == 0:
        normal_frac = float((in_lung == 0).sum()) / lung_n
        return normal_frac >= MIN_NORMAL_LUNG_FRAC
    n_cls = float((in_lung == target_cls).sum())
    if n_cls < MIN_PATHOLOGY_VOXELS: return False
    if (n_cls / lung_n) < MIN_TARGET_CLASS_FRAC: return False
    if PATCH_LABEL_MODE == 'dominant':
        dom = patch_dominant_label(seg_patch, lung_patch)
        if dom is None or int(dom) != int(target_cls): return False
    elif PATCH_LABEL_MODE == 'majority':
        labs = seg_patch[lung].astype(np.int64)
        vals, cnts = np.unique(labs, return_counts=True)
        maj = int(vals[int(np.argmax(cnts))])
        if maj != int(target_cls): return False
    return True

class VolumeCache:
    def __init__(self, data_dir, max_patients=None):
        self.data_dir = data_dir
        self.cache = {}
        self.max_patients = VOL_CACHE_PATIENTS if max_patients is None else max_patients
    def get(self, pid, path):
        if pid not in self.cache:
            if len(self.cache) >= self.max_patients:
                self.cache.pop(next(iter(self.cache)))
            packed = load_patient_volumes_3d(path, self.data_dir, inplane=INPLANE, pid_override=pid)
            if packed is not None:
                self.cache[pid] = packed
        return self.cache.get(pid)

def build_hierarchical_patch_bank(records, data_dir, n_patches=600, seed=42, min_per_class=None):
    """Build stratified patch bank with binary + hierarchical + 6-class labels."""
    floor_per_cls = MIN_PATCHES_PER_CLASS if min_per_class is None else int(min_per_class)
    rng = np.random.RandomState(seed)
    vol_cache = VolumeCache(data_dir)
    pid_to_rec = {r['pid']: r for r in records}
    bank = []
    seen_keys = set()

    # Count available pathology per patient (for stratification)
    ild_patients = [r for r in records if r['has_ild']]
    normal_patients = [r for r in records if not r['has_ild']]

    def collect_origins(seg_labels, lung_mask, cls_id):
        lung = lung_mask > 0.5
        if cls_id == 0:
            coords = np.argwhere((seg_labels == 0) & lung)
        else:
            coords = np.argwhere((seg_labels == cls_id) & lung)
        if len(coords) == 0: return []
        n_probe = min(len(coords), MAX_CLASS_ORIGIN_POOL * 4)
        if len(coords) > n_probe:
            pick = rng.choice(len(coords), size=n_probe, replace=False)
            coords = coords[pick]
        origins = []
        pd, ph, pw = CLS_PATCH_SIZE
        d, h, w = seg_labels.shape
        for cz, cy, cx in coords:
            oz = int(np.clip(cz - pd // 2, 0, max(0, d - pd)))
            oy = int(np.clip(cy - ph // 2, 0, max(0, h - ph)))
            ox = int(np.clip(cx - pw // 2, 0, max(0, w - pw)))
            origin = (oz, oy, ox)
            if origin in seen_keys: continue
            lung_crop = extract_patch(lung_mask, origin, CLS_PATCH_SIZE)
            seg_crop = extract_patch(seg_labels, origin, CLS_PATCH_SIZE)
            if not patch_label_ok(seg_crop, lung_crop, cls_id): continue
            origins.append(origin)
            if len(origins) >= MAX_CLASS_ORIGIN_POOL: break
        return origins

    # Stratify: 50% Normal, 50% ILD (balanced binary)
    binary_budget = n_patches
    normal_budget = binary_budget // 2
    ild_budget = binary_budget - normal_budget

    # Normal patches
    collected = 0
    rng.shuffle(normal_patients)
    for rec in normal_patients:
        if collected >= normal_budget: break
        packed = vol_cache.get(rec['pid'], rec['path'])
        if packed is None: continue
        origins = collect_origins(packed['seg_labels'], packed['lung_mask'], 0)
        for origin in origins:
            if collected >= normal_budget: break
            key = (rec['pid'], origin, 0)
            if key in seen_keys: continue
            seen_keys.add(key)
            bank.append(PatchRecord(pid=rec['pid'], origin=origin, label=0,
                                    hier_label=0, binary_label=0, group=rec['group']))
            collected += 1

    # ILD patches (from 5 pathology classes, stratified)
    pathology_collected = {c: 0 for c in range(1, 6)}
    ild_budget_per_class = ild_budget // 5
    for attempt in range(3):  # multiple rounds to fill quotas
        rng.shuffle(ild_patients)
        for rec in ild_patients:
            cc = rec['class_counts']
            for cls_id in range(1, 6):
                if cc[cls_id] == 0: continue
                if pathology_collected[cls_id] >= ild_budget_per_class: continue
                packed = vol_cache.get(rec['pid'], rec['path'])
                if packed is None: continue
                origins = collect_origins(packed['seg_labels'], packed['lung_mask'], cls_id)
                for origin in origins:
                    if pathology_collected[cls_id] >= ild_budget_per_class: break
                    key = (rec['pid'], origin, cls_id)
                    if key in seen_keys: continue
                    seen_keys.add(key)
                    hier_c = HIERARCHY_MAP[cls_id]
                    bank.append(PatchRecord(pid=rec['pid'], origin=origin, label=cls_id,
                                            hier_label=hier_c, binary_label=1, group=rec['group']))
                    pathology_collected[cls_id] += 1

    # Report class balance
    binary_counts = np.bincount([r.binary_label for r in bank], minlength=2)
    orig_counts = np.bincount([r.label for r in bank], minlength=6)
    print(f'  Binary: Normal={binary_counts[0]} ILD={binary_counts[1]}')
    print(f'  6-class: {dict(zip(ORIGINAL_CLASS_NAMES, orig_counts.tolist()))}')
    hier_counts = np.bincount([r.hier_label for r in bank], minlength=3)
    print(f'  Hierarchical: {dict(zip(HIERARCHY_CLASSES, hier_counts.tolist()))}')
    return bank, vol_cache

print('Stratified hierarchical patch mining defined.')
print(f'  Binary balance: 50/50 Normal/ILD')
print(f'  ILD sub-classes: stratified across 5 pathologies')

## 4 — Model Architecture

Shared 3D ResNet-18 encoder (MedicalNet pretrained) with three heads:
1. **Binary head**: Normal (0) vs ILD (1) — primary
2. **Hierarchical head**: Normal (0) / Fibrotic (1) / Non-fibrotic (2) — secondary
3. **5-class pathology head**: Emphysema / Fibrosis / Ground Glass / Micronodules / Consolidation — detailed mapping

In [ ]:
# ── Model Components ──
def _gn(ch):
    for g in (8, 4, 2, 1):
        if ch % g == 0: return nn.GroupNorm(g, ch)
    return nn.GroupNorm(1, ch)

class ResBlock3D(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = _gn(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = _gn(out_ch)
        self.skip = nn.Identity()
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                _gn(out_ch),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        return F.relu(out + self.skip(x), inplace=True)

class SEBlock3D(nn.Module):
    """Squeeze-and-Excitation block for 3D."""
    def __init__(self, channels, reduction=16):
        super().__init__()
        self.fc = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Conv3d(channels, channels // reduction, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv3d(channels // reduction, channels, kernel_size=1),
            nn.Sigmoid(),
        )
    def forward(self, x):
        return x * self.fc(x)

class SE_ResBlock3D(nn.Module):
    """ResBlock with squeeze-and-excitation."""
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv3d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = _gn(out_ch)
        self.conv2 = nn.Conv3d(out_ch, out_ch, 3, padding=1, bias=False)
        self.bn2 = _gn(out_ch)
        self.se = SEBlock3D(out_ch)
        self.skip = nn.Identity()
        if in_ch != out_ch or stride != 1:
            self.skip = nn.Sequential(
                nn.Conv3d(in_ch, out_ch, 1, stride=stride, bias=False),
                _gn(out_ch),
            )
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return F.relu(out + self.skip(x), inplace=True)

class HierarchicalEncoder3D(nn.Module):
    """Shared 3D ResNet-18 encoder with SE blocks + three classification heads.
    
    Heads:
      - binary_head: Normal (0) vs ILD (1)
      - hier_head: Normal (0) / Fibrotic (1) / Non-fibrotic (2)
      - path_head: 5-class pathology (Emphysema/Fibrosis/Ground Glass/Micronodules/Consolidation)
    """
    def __init__(self, in_ch=1, use_se=True):
        super().__init__()
        block = SE_ResBlock3D if use_se else ResBlock3D
        self.stem = nn.Sequential(
            nn.Conv3d(in_ch, 64, kernel_size=(1, 7, 7), stride=(1, 2, 2), padding=(0, 3, 3), bias=False),
            nn.GroupNorm(8, 64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(kernel_size=(1, 3, 3), stride=(1, 2, 2), padding=(0, 1, 1)),
        )
        self.layer1 = self._make_layer(64, 64, 2, block)
        self.layer2 = self._make_layer(64, 128, 2, block, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, block, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, block, stride=2)
        self.avgpool = nn.AdaptiveAvgPool3d((1, 1, 1))
        self.feat_dim = 512
        # Classification heads
        self.binary_head = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(self.feat_dim, N_BINARY_CLASSES),
        )
        self.hier_head = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(self.feat_dim, N_HIER_CLASSES),
        )
        self.path_head = nn.Sequential(
            nn.Dropout(HEAD_DROPOUT),
            nn.Linear(self.feat_dim, N_PATH_CLASSES),
        )
        self._pretrain_source = 'none'

    def _make_layer(self, in_ch, out_ch, blocks, block, stride=1):
        layers = [block(in_ch, out_ch, stride=stride)]
        for _ in range(1, blocks):
            layers.append(block(out_ch, out_ch))
        return nn.Sequential(*layers)

    def _ensure_med3d_weights(self):
        """Return path to Med3D weights, downloading from MED3D_URL if needed."""
        path = MED3D_WEIGHTS
        if path and os.path.isfile(path):
            return path
        if not MED3D_URL:
            return path
        dest = path if path else os.path.join(MODELS_DIR, 'resnet_18.pth')
        os.makedirs(os.path.dirname(os.path.abspath(dest)) or '.', exist_ok=True)
        if os.path.isfile(dest):
            return dest
        print(f'Downloading Med3D weights from {MED3D_URL}')
        print(f'  -> {dest}')
        try:
            urllib.request.urlretrieve(MED3D_URL, dest)
        except Exception as e:
            print(f'Med3D download failed: {e}')
            return path
        return dest if os.path.isfile(dest) else path

    def _load_pretrained_weights(self):
        weights_path = self._ensure_med3d_weights()
        if weights_path and os.path.isfile(weights_path):
            try:
                ckpt = torch.load(weights_path, map_location='cpu', weights_only=False)
                state = ckpt
                if isinstance(state, dict) and 'state_dict' in state:
                    state = state['state_dict']
                model_sd = self.state_dict()
                matched = 0
                for k, v in state.items():
                    nk = k
                    for pref in ('module.', 'backbone.', 'encoder.'):
                        if nk.startswith(pref): nk = nk[len(pref):]
                    nk = nk.replace('conv1', 'stem.0')
                    if nk in model_sd and model_sd[nk].shape == v.shape:
                        model_sd[nk] = v
                        matched += 1
                self.load_state_dict(model_sd, strict=False)
                self._pretrain_source = 'Med3D'
                print(f'Loaded Med3D weights from {weights_path} (matched {matched} tensors)')
                return
            except Exception as e:
                print(f'Med3D load failed: {e}')
                if REQUIRE_PRETRAIN:
                    raise RuntimeError(
                        f'ILD_REQUIRE_PRETRAIN=1 but failed to load Med3D weights at {weights_path}: {e}'
                    ) from e
        if REQUIRE_PRETRAIN:
            raise RuntimeError(
                'ILD_REQUIRE_PRETRAIN=1 but Med3D weights are missing. '
                f'Set ILD_MED3D_WEIGHTS (tried {MED3D_WEIGHTS!r}) or ensure ILD_MED3D_URL is reachable.'
            )
        # ImageNet inflate fallback (only when pretrain not required)
        try:
            import torchvision.models as tv_models
            model_2d = tv_models.resnet18(weights=tv_models.ResNet18_Weights.DEFAULT)
            sd_2d = model_2d.state_dict()
            w = sd_2d['conv1.weight']
            w3d = w.mean(dim=1, keepdim=True).unsqueeze(2)
            self.stem[0].weight.data.copy_(w3d)
            self._pretrain_source = 'ImageNet-inflate'
            print('Loaded ImageNet weights (2D->3D inflated)')
        except Exception as e:
            print(f'No pretrained weights: {e}')

    def extract_features(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        return self.avgpool(x).flatten(1)

    def forward(self, x, head='binary'):
        features = self.extract_features(x)
        if head == 'binary':
            return self.binary_head(features)
        elif head == 'hier':
            return self.hier_head(features)
        elif head == 'path':
            return self.path_head(features)
        else:
            return features

def set_trainable_blocks(encoder, unfreeze=UNFREEZE_BLOCKS):
    for prm in encoder.parameters():
        prm.requires_grad = False
    for name in unfreeze:
        mod = getattr(encoder, name, None)
        if mod is not None:
            for prm in mod.parameters():
                prm.requires_grad = True
    return encoder

# ── Loss Functions ──
class WeightedFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

class LabelSmoothCrossEntropy(nn.Module):
    def __init__(self, smoothing=LABEL_SMOOTHING):
        super().__init__()
        self.smoothing = smoothing
    def forward(self, logits, targets):
        n_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            smooth_targets = torch.full_like(log_probs, self.smoothing / (n_classes - 1))
            smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - self.smoothing)
        return (-smooth_targets * log_probs).sum(dim=1).mean()

def mixup_data(x, y, alpha=MIXUP_ALPHA):
    """MixUp augmentation for 3D patches."""
    if alpha <= 0: return x, y, None, None, 1.0
    lam = np.random.beta(alpha, alpha)
    batch_size = x.size(0)
    index = torch.randperm(batch_size).to(x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    return mixed_x, y, y[index], lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

# Instantiate model
model = HierarchicalEncoder3D(in_ch=1, use_se=True).to(device)
model._load_pretrained_weights()
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'HierarchicalEncoder3D: {total_params:,} total, {trainable_params:,} trainable')
print(f'  Binary head: 2 classes | Hier head: 3 classes | Path head: 5 classes')
print(f'  SE blocks: enabled | MixUp alpha: {MIXUP_ALPHA} | Label smoothing: {LABEL_SMOOTHING}')


## 5 — Training (Repeated 5-Fold GroupKFold)

In [ ]:
def cuda_cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def train_fold(train_recs, val_recs, fold_idx, repeat_idx):
    """Train one fold, return metrics for all three heads."""
    fold_name = f'R{repeat_idx}F{fold_idx}'
    print(f'\n=== {fold_name}: train={len(train_recs)} val={len(val_recs)} patients ===')

    # Build patch banks
    train_bank, train_vc = build_hierarchical_patch_bank(
        train_recs, MEDGIFT_ROOT, n_patches=PATCHES_PER_EPOCH,
        seed=GLOBAL_SEED + repeat_idx * 100 + fold_idx)
    val_bank, val_vc = build_hierarchical_patch_bank(
        val_recs, MEDGIFT_ROOT, n_patches=VAL_PATCHES,
        seed=GLOBAL_SEED + 999 + repeat_idx * 100 + fold_idx,
        min_per_class=VAL_PATCHES // 6)

    # Class weights
    bin_counts = np.bincount([r.binary_label for r in train_bank], minlength=2).astype(float)
    bin_freq = bin_counts / max(bin_counts.sum(), 1)
    bin_alpha = torch.tensor(1.0 / np.maximum(bin_freq, 1e-6), dtype=torch.float32).to(device)
    bin_alpha = bin_alpha / bin_alpha.sum() * 2

    hier_counts = np.bincount([r.hier_label for r in train_bank], minlength=3).astype(float)
    hier_freq = hier_counts / max(hier_counts.sum(), 1)
    hier_alpha = torch.tensor(1.0 / np.maximum(hier_freq, 1e-6), dtype=torch.float32).to(device)
    hier_alpha = hier_alpha / hier_alpha.sum() * 3

    path_mask = np.array([r.label for r in train_bank]) > 0
    if path_mask.sum() > 0:
        path_counts = np.bincount(
            [r.label - 1 for r in train_bank if r.label > 0], minlength=5).astype(float)
        path_freq = path_counts / max(path_counts.sum(), 1)
        path_alpha = torch.tensor(1.0 / np.maximum(path_freq, 1e-6), dtype=torch.float32).to(device)
        path_alpha = path_alpha / path_alpha.sum() * 5
    else:
        path_alpha = None

    # Dataset
    class PatchDataset(Dataset):
        def __init__(self, bank, vol_cache, pid_to_rec, augment=True):
            self.bank = bank
            self.vol_cache = vol_cache
            self.pid_to_rec = pid_to_rec
            self.augment = augment
        def __len__(self): return len(self.bank)
        def __getitem__(self, idx):
            rec = self.bank[idx]
            packed = self.vol_cache.get(rec.pid, self.pid_to_rec[rec.pid]['path'])
            if packed is None:
                x = torch.zeros(1, *CLS_PATCH_SIZE)
                return x, torch.tensor(0), torch.tensor(0), torch.tensor(0)
            ct = packed['ct_norm']
            x = extract_patch(ct, rec.origin, CLS_PATCH_SIZE)
            x = torch.from_numpy(x).unsqueeze(0).float()
            # Augmentations
            if self.augment:
                if torch.rand(1).item() > 0.5: x = x.flip(-1)
                if torch.rand(1).item() > 0.5: x = x.flip(-2)
                if torch.rand(1).item() > 0.5:
                    k = int(torch.randint(0, 4, (1,)).item())
                    x = torch.rot90(x, k, dims=(-2, -1))
                if torch.rand(1).item() > 0.5:
                    scale = float(torch.empty(1).uniform_(0.9, 1.1).item())
                    shift = float(torch.empty(1).uniform_(-0.05, 0.05).item())
                    x = (x * scale + shift).clamp(0.0, 1.0)
            return x, torch.tensor(rec.binary_label, dtype=torch.long), \
                   torch.tensor(rec.hier_label, dtype=torch.long), \
                   torch.tensor(rec.label, dtype=torch.long)

    pid_to_rec_tr = {r['pid']: r for r in train_recs}
    pid_to_rec_val = {r['pid']: r for r in val_recs}
    train_ds = PatchDataset(train_bank, train_vc, pid_to_rec_tr, augment=PATCH_AUGMENT)
    val_ds = PatchDataset(val_bank, val_vc, pid_to_rec_val, augment=False)
    train_loader = DataLoader(train_ds, batch_size=FEAT_BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    val_loader = DataLoader(val_ds, batch_size=FEAT_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    # Initialize model for this fold
    model = HierarchicalEncoder3D(in_ch=1, use_se=True).to(device)
    model._load_pretrained_weights()

    # Loss functions
    bin_criterion = WeightedFocalLoss(alpha=bin_alpha, gamma=2.0)
    hier_criterion = LabelSmoothCrossEntropy(smoothing=LABEL_SMOOTHING)
    path_criterion = WeightedFocalLoss(alpha=path_alpha, gamma=2.0) if path_alpha is not None else nn.CrossEntropyLoss()

    def make_discriminative_optimizer():
        """No-probe schedule: low LR on early blocks, higher on layer4 + binary head."""
        for prm in model.parameters():
            prm.requires_grad = True
        # Freeze secondary heads during binary-primary training
        if BINARY_ONLY_TRAIN:
            for head in (model.hier_head, model.path_head):
                for prm in head.parameters():
                    prm.requires_grad = False
        groups = [
            {'params': list(model.stem.parameters()) + list(model.layer1.parameters())
             + list(model.layer2.parameters()), 'lr': LR_STEM},
            {'params': list(model.layer3.parameters()), 'lr': LR_LAYER3},
            {'params': list(model.layer4.parameters()), 'lr': LR_LAYER4},
            {'params': list(model.binary_head.parameters()), 'lr': LR_HEAD},
        ]
        if not BINARY_ONLY_TRAIN:
            groups.append({'params': list(model.hier_head.parameters())
                           + list(model.path_head.parameters()), 'lr': LR_HEAD})
        # Drop empty groups
        groups = [g for g in groups if len(list(g['params'])) > 0]
        return torch.optim.AdamW(groups, weight_decay=FINETUNE_WD)

    def make_optimizer(phase):
        if phase == 'probe':
            for prm in model.parameters():
                prm.requires_grad = False
            heads = [model.binary_head] if BINARY_ONLY_TRAIN else [
                model.binary_head, model.hier_head, model.path_head]
            for head in heads:
                for prm in head.parameters():
                    prm.requires_grad = True
            return torch.optim.AdamW(
                [{'params': h.parameters()} for h in heads],
                lr=PROBE_LR, weight_decay=FINETUNE_WD)
        if WARMUP_EPOCHS <= 0:
            return make_discriminative_optimizer()
        set_trainable_blocks(model, UNFREEZE_BLOCKS)
        for prm in model.binary_head.parameters():
            prm.requires_grad = True
        if BINARY_ONLY_TRAIN:
            for head in (model.hier_head, model.path_head):
                for prm in head.parameters():
                    prm.requires_grad = False
        enc_params = [p for n, p in model.named_parameters()
                      if p.requires_grad and not n.startswith(('binary_head', 'hier_head', 'path_head'))]
        groups = [
            {'params': enc_params, 'lr': FINETUNE_LR},
            {'params': model.binary_head.parameters(), 'lr': FINETUNE_HEAD_LR},
        ]
        if not BINARY_ONLY_TRAIN:
            groups += [
                {'params': model.hier_head.parameters(), 'lr': FINETUNE_HEAD_LR},
                {'params': model.path_head.parameters(), 'lr': FINETUNE_HEAD_LR},
            ]
        return torch.optim.AdamW(groups, weight_decay=FINETUNE_WD)

    def make_scheduler(opt, phase):
        t_max = max(1, ENCODER_EPOCHS if WARMUP_EPOCHS <= 0 else (
            WARMUP_EPOCHS if phase == 'probe' else ENCODER_EPOCHS - WARMUP_EPOCHS))
        return torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=t_max)

    phase = 'probe' if WARMUP_EPOCHS > 0 else 'disc'
    optimizer = make_optimizer(phase)
    scheduler = make_scheduler(optimizer, phase)
    print(f'  Train mode: phase={phase} BINARY_ONLY={BINARY_ONLY_TRAIN} '
          f'patience={FINETUNE_PATIENCE}')

    best_bin_f1 = -1.0
    best_state = None
    stale = 0
    history = []

    for epoch in range(ENCODER_EPOCHS):
        if phase == 'probe' and epoch >= WARMUP_EPOCHS:
            phase = 'finetune'
            optimizer = make_optimizer(phase)
            scheduler = make_scheduler(optimizer, phase)
            stale = 0
            print(f'  -> Phase B (fine-tune) epoch {epoch+1}')

        model.train()
        total_loss = 0.0
        n_batches = 0
        optimizer.zero_grad(set_to_none=True)
        use_mixup = MIXUP_ALPHA > 0  # MixUp from epoch 1 (no Phase-B-only shock)
        for step, (x, y_bin, y_hier, y_orig) in enumerate(train_loader):
            x, y_bin, y_hier, y_orig = x.to(device), y_bin.to(device), y_hier.to(device), y_orig.to(device)
            mixed = False
            if use_mixup:
                x, y_bin_a, y_bin_b, lam = mixup_data(x, y_bin, alpha=MIXUP_ALPHA)
                mixed = True
            feat = model.extract_features(x)
            bin_logits = model.binary_head(feat)
            if mixed:
                bin_loss = mixup_criterion(bin_criterion, bin_logits, y_bin_a, y_bin_b, lam)
            else:
                bin_loss = bin_criterion(bin_logits, y_bin)
            if BINARY_ONLY_TRAIN:
                loss = bin_loss / GRAD_ACCUM_STEPS
            else:
                hier_logits = model.hier_head(feat)
                hier_loss = hier_criterion(hier_logits, y_hier)
                path_mask = y_orig > 0
                if path_mask.sum() > 0:
                    path_logits = model.path_head(feat[path_mask])
                    path_loss = path_criterion(path_logits, y_orig[path_mask] - 1)
                else:
                    path_loss = torch.tensor(0.0, device=device)
                loss = (bin_loss + 0.5 * hier_loss + 0.3 * path_loss) / GRAD_ACCUM_STEPS
            loss.backward()
            if ((step + 1) % GRAD_ACCUM_STEPS == 0) or ((step + 1) == len(train_loader)):
                optimizer.step()
                optimizer.zero_grad(set_to_none=True)
            total_loss += loss.item() * GRAD_ACCUM_STEPS
            n_batches += 1

        train_loss = total_loss / max(n_batches, 1)

        # Validation
        model.eval()
        all_bin_true, all_bin_pred, all_bin_prob = [], [], []
        all_hier_true, all_hier_pred = [], []
        all_path_true, all_path_pred = [], []
        with torch.no_grad():
            for x, y_bin, y_hier, y_orig in val_loader:
                x = x.to(device)
                feat = model.extract_features(x)
                bin_logits = model.binary_head(feat)
                bin_probs = F.softmax(bin_logits, dim=1)
                bin_preds = bin_probs.argmax(1)
                all_bin_true.extend(y_bin.numpy().tolist())
                all_bin_pred.extend(bin_preds.cpu().numpy().tolist())
                all_bin_prob.append(bin_probs.cpu().numpy())
                hier_logits = model.hier_head(feat)
                hier_preds = hier_logits.argmax(1)
                all_hier_true.extend(y_hier.numpy().tolist())
                all_hier_pred.extend(hier_preds.cpu().numpy().tolist())
                path_mask = y_orig > 0
                if path_mask.sum() > 0:
                    path_logits = model.path_head(feat[path_mask.to(device)])
                    path_preds = path_logits.argmax(1)
                    all_path_true.extend((y_orig[path_mask] - 1).numpy().tolist())
                    all_path_pred.extend(path_preds.cpu().numpy().tolist())

        all_bin_prob = np.concatenate(all_bin_prob) if all_bin_prob else np.array([])
        bin_f1 = f1_score(all_bin_true, all_bin_pred, average='binary', zero_division=0)
        hier_f1 = f1_score(all_hier_true, all_hier_pred, average='macro', zero_division=0)
        hier_wf1 = f1_score(all_hier_true, all_hier_pred, average='weighted', zero_division=0)
        path_f1 = f1_score(all_path_true, all_path_pred, average='macro', zero_division=0) if all_path_true else 0.0
        path_wf1 = f1_score(all_path_true, all_path_pred, average='weighted', zero_division=0) if all_path_true else 0.0

        scheduler.step()

        history.append({
            'epoch': epoch + 1, 'phase': phase,
            'train_loss': train_loss,
            'bin_f1': float(bin_f1),
            'hier_f1': float(hier_f1),
            'path_f1': float(path_f1),
        })

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'  Ep {epoch+1}/{ENCODER_EPOCHS} [{phase}] loss={train_loss:.4f} '
                  f'binF1={bin_f1:.4f} hierF1={hier_f1:.4f} pathF1={path_f1:.4f}'
                  f'{" [bin-only]" if BINARY_ONLY_TRAIN else ""}')

        if bin_f1 > best_bin_f1:
            best_bin_f1 = bin_f1
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            stale = 0
        else:
            stale += 1
            # Patience from epoch 1 (no probe→finetune reset advantage)
            if stale >= FINETUNE_PATIENCE:
                print(f'  Early stop at epoch {epoch+1} (patience={FINETUNE_PATIENCE})')
                break

    # Restore best state
    if best_state:
        model.load_state_dict(best_state)

    # Full evaluation
    model.eval()
    all_bin_true, all_bin_pred, all_bin_prob = [], [], []
    all_hier_true, all_hier_pred, all_hier_prob = [], [], []
    all_path_true, all_path_pred, all_path_prob = [], [], []
    all_orig_true = []
    with torch.no_grad():
        for x, y_bin, y_hier, y_orig in val_loader:
            x = x.to(device)
            feat = model.extract_features(x)
            bin_logits = model.binary_head(feat)
            bin_probs = F.softmax(bin_logits, dim=1)
            bin_preds = bin_probs.argmax(1)
            all_bin_true.extend(y_bin.numpy().tolist())
            all_bin_pred.extend(bin_preds.cpu().numpy().tolist())
            all_bin_prob.append(bin_probs.cpu().numpy())
            hier_logits = model.hier_head(feat)
            hier_probs = F.softmax(hier_logits, dim=1)
            hier_preds = hier_probs.argmax(1)
            all_hier_true.extend(y_hier.numpy().tolist())
            all_hier_pred.extend(hier_preds.cpu().numpy().tolist())
            all_hier_prob.append(hier_probs.cpu().numpy())
            all_orig_true.extend(y_orig.numpy().tolist())
            path_mask = y_orig > 0
            if path_mask.sum() > 0:
                path_logits = model.path_head(feat[path_mask.to(device)])
                path_probs = F.softmax(path_logits, dim=1)
                path_preds = path_probs.argmax(1)
                all_path_true.extend((y_orig[path_mask] - 1).numpy().tolist())
                all_path_pred.extend(path_preds.cpu().numpy().tolist())
                all_path_prob.append(path_probs.cpu().numpy())

    all_bin_prob = np.concatenate(all_bin_prob) if all_bin_prob else np.array([])
    all_hier_prob = np.concatenate(all_hier_prob) if all_hier_prob else np.array([])
    all_path_prob = np.concatenate(all_path_prob) if all_path_prob else np.array([])

    # Metrics
    bin_metrics = {}
    if len(all_bin_true) > 0:
        bin_metrics = {
            'accuracy': float(accuracy_score(all_bin_true, all_bin_pred)),
            'precision': float(precision_score(all_bin_true, all_bin_pred, zero_division=0)),
            'recall': float(recall_score(all_bin_true, all_bin_pred, zero_division=0)),
            'f1': float(f1_score(all_bin_true, all_bin_pred, zero_division=0)),
            'mcc': float(matthews_corrcoef(all_bin_true, all_bin_pred)),
            'kappa': float(cohen_kappa_score(all_bin_true, all_bin_pred)),
            'n_val': len(all_bin_true),
        }
        if len(np.unique(all_bin_true)) > 1:
            bin_metrics['auc_roc'] = float(roc_auc_score(all_bin_true, all_bin_prob[:, 1]))

    hier_metrics = {}
    if len(all_hier_true) > 0:
        hier_metrics = {
            'accuracy': float(accuracy_score(all_hier_true, all_hier_pred)),
            'f1_macro': float(f1_score(all_hier_true, all_hier_pred, average='macro', zero_division=0)),
            'f1_weighted': float(f1_score(all_hier_true, all_hier_pred, average='weighted', zero_division=0)),
            'n_val': len(all_hier_true),
        }
        try:
            hier_metrics['auc_macro'] = float(roc_auc_score(
                all_hier_true, all_hier_prob, multi_class='ovr', average='macro',
                labels=list(range(N_HIER_CLASSES))))
        except Exception:
            hier_metrics['auc_macro'] = float('nan')

    path_metrics = {}
    if len(all_path_true) > 0:
        path_metrics = {
            'accuracy': float(accuracy_score(all_path_true, all_path_pred)),
            'f1_macro': float(f1_score(all_path_true, all_path_pred, average='macro', zero_division=0)),
            'f1_weighted': float(f1_score(all_path_true, all_path_pred, average='weighted', zero_division=0)),
            'n_val': len(all_path_true),
        }

    # Save probabilities for bootstrap
    fold_tag = f'{fold_name}'
    np.savez(os.path.join(EXPORTS_DIR, f'hier_val_probs_{fold_tag}.npz'),
             y_bin=np.array(all_bin_true, dtype=np.int64),
             probs_bin=all_bin_prob.astype(np.float32),
             y_hier=np.array(all_hier_true, dtype=np.int64),
             probs_hier=all_hier_prob.astype(np.float32))

    # Save model checkpoint
    torch.save({
        'model': model.state_dict(),
        'best_bin_f1': best_bin_f1,
        'fold': fold_idx, 'repeat': repeat_idx,
    }, os.path.join(MODELS_DIR, f'hierarchical_fold_{fold_tag}.pth'))

    cuda_cleanup()
    return bin_metrics, hier_metrics, path_metrics, history

# ── Repeated GroupKFold ──
all_results = []
all_histories = []

if not RUN_TRAIN:
    print('SKIP training: ILD_RUN_TRAIN=0. Load existing fold metrics/probs if present.')
    cv_path = os.path.join(EXPORTS_DIR, 'hierarchical_cv_results.json')
    if os.path.isfile(cv_path):
        with open(cv_path) as f:
            cv_results = json.load(f)
        all_results = cv_results.get('folds', [])
        print(f'Loaded {len(all_results)} folds from {cv_path}')
    else:
        print(f'No cached results at {cv_path}; downstream cells may be empty.')

for repeat in range(N_REPEATS if RUN_TRAIN else 0):
    seed = GLOBAL_SEED + repeat * 1000
    rng = np.random.RandomState(seed)
    groups = list(set(r['group'] for r in patient_records))
    rng.shuffle(groups)
    
    # Use StratifiedKFold on patient labels to ensure ILD/Normal balance per fold
    group_has_ild = []
    for g in groups:
        recs_for_g = [r for r in patient_records if r['group'] == g]
        group_has_ild.append(1 if any(r['has_ild'] for r in recs_for_g) else 0)
    
    n_splits = min(N_FOLDS, len(groups))
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    for fold_idx, (train_idx, val_idx) in enumerate(skf.split(groups, group_has_ild)):
        train_groups = [groups[i] for i in train_idx]
        val_groups = [groups[i] for i in val_idx]
        train_recs = [r for r in patient_records if r['group'] in train_groups]
        val_recs = [r for r in patient_records if r['group'] in val_groups]
        
        bin_met, hier_met, path_met, hist = train_fold(
            train_recs, val_recs, fold_idx, repeat)
        all_results.append({
            'repeat': repeat, 'fold': fold_idx,
            'binary': bin_met,
            'hierarchical': hier_met,
            'pathology': path_met,
        })
        all_histories.append({'repeat': repeat, 'fold': fold_idx, 'history': hist})

# ── Aggregate Results ──
print('\n' + '=' * 60)
print('HIERARCHICAL ILD-XR: CROSS-VALIDATION RESULTS')
print('=' * 60)
print(f'Repeated {N_REPEATS}x{N_FOLDS}-fold CV: {len(all_results)} total folds | RUN_TRAIN={RUN_TRAIN}')

if not all_results:
    print('No fold results to aggregate.')
    cv_results = {
        'n_repeats': N_REPEATS,
        'n_folds': N_FOLDS,
        'n_total_folds': 0,
        'binary_summary': {},
        'hier_summary': {},
        'path_summary': {},
        'folds': [],
        'skipped_train': not RUN_TRAIN,
    }
bin_f1s = [r['binary'].get('f1', 0) for r in all_results if r.get('binary')]
bin_aucs = [r['binary'].get('auc_roc', 0) for r in all_results if r['binary'] and 'auc_roc' in r['binary']]
hier_f1s = [r['hierarchical'].get('f1_macro', 0) for r in all_results if r['hierarchical']]
hier_wf1s = [r['hierarchical'].get('f1_weighted', 0) for r in all_results if r['hierarchical']]
path_f1s = [r['pathology'].get('f1_macro', 0) for r in all_results if r['pathology']]
path_wf1s = [r['pathology'].get('f1_weighted', 0) for r in all_results if r['pathology']]
bin_accs = [r['binary'].get('accuracy', 0) for r in all_results if r['binary']]
bin_mccs = [r['binary'].get('mcc', 0) for r in all_results if r['binary']]

def _ms(vals):
    return (float(np.mean(vals)), float(np.std(vals))) if vals else (float('nan'), float('nan'))

if all_results:
    print(f'\nPrimary (Binary ILD Detection):')
    m, s = _ms(bin_f1s); print(f'  F1        = {m:.4f} ± {s:.4f}')
    m, s = _ms(bin_aucs); print(f'  AUC-ROC   = {m:.4f} ± {s:.4f}')
    m, s = _ms(bin_accs); print(f'  Accuracy  = {m:.4f} ± {s:.4f}')
    m, s = _ms(bin_mccs); print(f'  MCC       = {m:.4f} ± {s:.4f}')

    print(f'\nSecondary (3-Class Hierarchical):')
    m, s = _ms(hier_f1s); print(f'  Macro-F1   = {m:.4f} ± {s:.4f}')
    m, s = _ms(hier_wf1s); print(f'  Weighted-F1= {m:.4f} ± {s:.4f}')

    print(f'\nTertiary (5-Class Pathology):')
    m, s = _ms(path_f1s); print(f'  Macro-F1   = {m:.4f} ± {s:.4f}')
    m, s = _ms(path_wf1s); print(f'  Weighted-F1= {m:.4f} ± {s:.4f}')

# Save raw results (overwrite only when we have folds or trained this run)
cv_results = {
    'n_repeats': N_REPEATS,
    'n_folds': N_FOLDS,
    'n_total_folds': len(all_results),
    'binary_summary': {
        'f1_mean': _ms(bin_f1s)[0],
        'f1_std': _ms(bin_f1s)[1],
        'auc_mean': _ms(bin_aucs)[0],
        'auc_std': _ms(bin_aucs)[1],
        'accuracy_mean': _ms(bin_accs)[0],
        'mcc_mean': _ms(bin_mccs)[0],
    },
    'hier_summary': {
        'f1_macro_mean': _ms(hier_f1s)[0],
        'f1_macro_std': _ms(hier_f1s)[1],
        'f1_weighted_mean': _ms(hier_wf1s)[0],
    },
    'path_summary': {
        'f1_macro_mean': _ms(path_f1s)[0],
        'f1_macro_std': _ms(path_f1s)[1],
        'f1_weighted_mean': _ms(path_wf1s)[0],
    },
    'folds': all_results,
}
with open(os.path.join(EXPORTS_DIR, 'hierarchical_cv_results.json'), 'w') as f:
    json.dump(cv_results, f, indent=2)
print(f'\nSaved: {EXPORTS_DIR}/hierarchical_cv_results.json')


## 5b — Full-Volume Cascade Inference

**Current proposal (cascade), not literature U-Net then separate classifier:**

`CT -> lungmask (cached) -> patch classifier (sliding window) -> full 3D pathology map -> [patient classification + volumetric biomarkers]`

Dense Softmax voting over lung-masked tissue (methodology Inference section): stride default `(4,8,8)`, count-weighted posterior accumulation, argmax labels, median filter. Hierarchical heads fuse into a 6-class map:

`P(Normal)=P_bin(Normal)`, `P(class c)=P_bin(ILD)*P_path(c)` for pathology classes.


In [ ]:
# Full-volume cascade: patch Softmax -> 3D pathology map -> patient labels + biomarkers
from scipy.ndimage import median_filter

CASCADE_DIR = os.path.join(EXPORTS_DIR, 'cascade_maps')
os.makedirs(CASCADE_DIR, exist_ok=True)

def _resolve_hier_checkpoint():
    """Prefer best binary-F1 fold from this run; else any hierarchical_fold_*.pth."""
    best_path, best_f1 = None, -1.0
    names = sorted(os.listdir(MODELS_DIR)) if os.path.isdir(MODELS_DIR) else []
    for name in names:
        if not (name.startswith('hierarchical_fold_') and name.endswith('.pth')):
            continue
        path = os.path.join(MODELS_DIR, name)
        try:
            ck = torch.load(path, map_location='cpu', weights_only=False)
            f1 = float(ck.get('best_bin_f1', -1))
            if f1 > best_f1:
                best_f1, best_path = f1, path
        except Exception:
            if best_path is None:
                best_path = path
    return best_path, best_f1


def cascade_classify_volume(ct_norm, lung_mask, model, patch_size=None, stride=None, max_patches=None, path_thresh=None):
    """Sliding-window hierarchical Softmax -> 6-class volume map + patient summary.

    Matches proposal: CT -> patch classifier -> full 3D pathology map.
    """
    if patch_size is None:
        patch_size = CLS_PATCH_SIZE
    if stride is None:
        stride = INFER_DENSE_STRIDE
    if max_patches is None:
        max_patches = INFER_MAX_PATCHES
    if path_thresh is None:
        path_thresh = float(os.environ.get('ILD_CASCADE_PATH_THRESH', '0.05'))

    D, H, W = ct_norm.shape
    pd, ph, pw = patch_size
    n_cls = SEG_NUM_CLASSES  # 6 = Normal + 5 pathology
    vote = np.zeros((n_cls, D, H, W), dtype=np.float32)
    weight = np.zeros((D, H, W), dtype=np.float32)

    coords = np.argwhere(lung_mask > 0.5)
    if len(coords) == 0:
        empty = np.zeros((D, H, W), dtype=np.int64)
        return empty, {'n_patches': 0, 'pred_binary': 0, 'pred_path': 0, 'pred_hier': 0}

    z0, y0, x0 = coords.min(axis=0)
    z1, y1, x1 = coords.max(axis=0) + 1
    zs = list(range(max(0, z0 - pd // 2), max(1, min(D, z1) - pd + 1), stride[0])) or [0]
    ys = list(range(max(0, y0 - ph // 2), max(1, min(H, y1) - ph + 1), stride[1])) or [0]
    xs = list(range(max(0, x0 - pw // 2), max(1, min(W, x1) - pw + 1), stride[2])) or [0]

    n = 0
    model.eval()
    with torch.no_grad():
        for oz in zs:
            for oy in ys:
                for ox in xs:
                    if n >= max_patches:
                        break
                    origin = (oz, oy, ox)
                    lung_crop = extract_patch(lung_mask, origin, patch_size)
                    if float(lung_crop.mean()) < MIN_PATCH_LUNG_FRAC:
                        continue
                    patch = extract_patch(ct_norm, origin, patch_size)
                    x = torch.from_numpy(patch).unsqueeze(0).unsqueeze(0).float().to(device)
                    feat = model.extract_features(x)
                    p_bin = F.softmax(model.binary_head(feat), dim=1)[0].cpu().numpy()
                    p_path = F.softmax(model.path_head(feat), dim=1)[0].cpu().numpy()
                    # Hierarchical fuse -> 6-class posterior
                    proba6 = np.zeros(n_cls, dtype=np.float32)
                    proba6[0] = float(p_bin[0])
                    proba6[1:] = float(p_bin[1]) * p_path
                    s = float(proba6.sum()) + 1e-8
                    proba6 /= s

                    d, h, w = lung_crop.shape
                    # Clamp to actual slice dims (patch may extend past volume boundary)
                    ed = min(oz + d, D) - oz
                    eh = min(oy + h, H) - oy
                    ew = min(ox + w, W) - ox
                    m = lung_crop[:ed, :eh, :ew] > 0.5
                    # Use direct slice views (contiguous slices support in-place +=)
                    vsub = vote[:, oz:oz + ed, oy:oy + eh, ox:ox + ew]  # (n_cls, ed, eh, ew)
                    vsub[:, m] += proba6[:, np.newaxis]  # (n_cls, N) += (n_cls, 1)
                    vote[:, oz:oz + ed, oy:oy + eh, ox:ox + ew] = vsub
                    wsub = weight[oz:oz + ed, oy:oy + eh, ox:ox + ew]
                    wsub[m] += 1.0
                    weight[oz:oz + ed, oy:oy + eh, ox:ox + ew] = wsub
                    n += 1
                    if n % INFER_CLEANUP_EVERY == 0:
                        cuda_cleanup()

    weight = np.maximum(weight, 1e-6)
    vol = np.zeros((D, H, W), dtype=np.int64)
    inside = lung_mask > 0.5
    if inside.any():
        probs = vote[:, inside] / weight[inside]
        vol[inside] = np.argmax(probs, axis=0)
        vol_f = median_filter(vol.astype(np.float32), size=3)
        vol = np.where(inside, np.rint(vol_f).astype(np.int64), 0)
        vol = np.clip(vol, 0, n_cls - 1)

    hist = np.bincount(vol[inside].ravel(), minlength=n_cls) if inside.any() else np.zeros(n_cls, dtype=np.int64)
    lung_vox = int(inside.sum())
    path_vox = int(hist[1:].sum())
    path_frac = float(path_vox / max(lung_vox, 1))
    # Soft mean-ILD-probability over lung voxels (more robust than argmax voxel count)
    if inside.any() and weight[inside].max() > 0:
        mean_ild_prob = float((vote[1:, inside] / weight[inside]).sum(axis=0).mean())
    else:
        mean_ild_prob = 0.0
    # Patient is ILD if pathology fraction >= threshold OR soft ILD prob >= 0.45
    pred_binary = 1 if (path_frac >= path_thresh or mean_ild_prob >= 0.45) else 0
    pred_path = int(np.argmax(hist[1:]) + 1) if path_vox > 0 else 0
    fibro = int(hist[2] + hist[5])
    nonfib = int(hist[1] + hist[3] + hist[4])
    if pred_binary == 0:
        pred_hier = 0
    elif fibro >= nonfib:
        pred_hier = 1
    else:
        pred_hier = 2

    meta = {
        'n_patches': int(n),
        'pred_binary': pred_binary,
        'pred_path': pred_path,
        'pred_hier': pred_hier,
        'hist': hist.astype(int).tolist(),
        'lung_voxels': lung_vox,
        'pathology_voxels': path_vox,
        'pathology_frac': path_frac,
        'mean_ild_prob': mean_ild_prob,
        'fibrotic_frac': float(fibro / max(lung_vox, 1)),
        'nonfibrotic_frac': float(nonfib / max(lung_vox, 1)),
    }
    return vol, meta


def run_full_volume_cascade(records, ckpt_path=None, max_patients=0):
    """CT -> sliding-window patch Softmax -> 3D map -> patient classification/biomarkers."""
    if ckpt_path is None:
        ckpt_path, best_f1 = _resolve_hier_checkpoint()
        print(f'Cascade checkpoint: {ckpt_path} (best_bin_f1={best_f1})')
    else:
        best_f1 = None
        print(f'Cascade checkpoint: {ckpt_path}')
    if not ckpt_path or not os.path.isfile(ckpt_path):
        print('SKIP cascade: no hierarchical_fold_*.pth found (train first).')
        return None

    model = HierarchicalEncoder3D(in_ch=1, use_se=True).to(device)
    ck = torch.load(ckpt_path, map_location=device, weights_only=False)
    state = ck.get('model', ck)
    model.load_state_dict(state, strict=False)
    model.eval()

    recs = list(records)
    if max_patients and max_patients > 0:
        recs = recs[:max_patients]

    rows = []
    vc = VolumeCache(MEDGIFT_ROOT, max_patients=2)
    for rec in tqdm(recs, desc='Full-volume cascade'):
        packed = vc.get(rec['pid'], rec['path'])
        if packed is None:
            continue
        ct = packed['ct_norm']
        lung = packed['lung_mask']
        vol_map, meta = cascade_classify_volume(ct, lung, model)

        gt_seg = packed.get('seg_labels')
        gt_binary = int(rec.get('has_ild', 0))
        gt_path = -1
        if gt_seg is not None:
            lung_b = lung > 0.5
            counts = (
                np.bincount(gt_seg[lung_b].astype(np.int64), minlength=SEG_NUM_CLASSES)
                if lung_b.any() else np.zeros(SEG_NUM_CLASSES)
            )
            gt_path = int(np.argmax(counts[1:]) + 1) if counts[1:].sum() > 0 else 0

        if CASCADE_SAVE_MAPS:
            np.savez_compressed(
                os.path.join(CASCADE_DIR, f"map_{rec['pid']}.npz"),
                pathology_map=vol_map.astype(np.int16),
                lung_mask=(lung > 0.5).astype(np.uint8),
                hist=np.array(meta['hist'], dtype=np.int64),
            )

        rows.append({
            'pid': rec['pid'],
            'group': rec['group'],
            'cohort': rec.get('cohort'),
            'gt_binary': gt_binary,
            'pred_binary': meta['pred_binary'],
            'gt_path': gt_path,
            'pred_path': meta['pred_path'],
            'pred_hier': meta['pred_hier'],
            'n_patches': meta['n_patches'],
            'lung_voxels': meta['lung_voxels'],
            'pathology_voxels': meta['pathology_voxels'],
            'pathology_frac': meta['pathology_frac'],
            'fibrotic_frac': meta['fibrotic_frac'],
            'nonfibrotic_frac': meta['nonfibrotic_frac'],
            **{
                f'vox_{ORIGINAL_CLASS_NAMES[i].replace(" ", "_")}': meta['hist'][i]
                for i in range(SEG_NUM_CLASSES)
            },
        })
        del packed, vol_map
        cuda_cleanup()

    del model
    cuda_cleanup()

    df = pd.DataFrame(rows)
    out_csv = os.path.join(EXPORTS_DIR, 'hierarchical_cascade_patients.csv')
    df.to_csv(out_csv, index=False)

    summary = {
        'protocol': 'CT_lungmask_sliding_window_patch_softmax_3D_map_then_classify_biomarkers',
        'proposal': 'cascade_not_unet_then_separate_classifier',
        'checkpoint': ckpt_path,
        'best_bin_f1_ckpt': best_f1,
        'stride': list(INFER_DENSE_STRIDE),
        'patch_size': list(CLS_PATCH_SIZE),
        'n_patients': int(len(df)),
        'maps_dir': CASCADE_DIR if CASCADE_SAVE_MAPS else None,
    }
    if len(df) and 'gt_binary' in df.columns:
        yb, pb = df['gt_binary'].astype(int), df['pred_binary'].astype(int)
        summary['patient_binary_f1'] = float(f1_score(yb, pb, zero_division=0))
        summary['patient_binary_acc'] = float(accuracy_score(yb, pb))
        mask = df['gt_path'] > 0
        if mask.any():
            summary['patient_path_macro_f1'] = float(f1_score(
                df.loc[mask, 'gt_path'], df.loc[mask, 'pred_path'],
                labels=list(range(1, 6)), average='macro', zero_division=0,
            ))

    out_json = os.path.join(EXPORTS_DIR, 'hierarchical_cascade_summary.json')
    with open(out_json, 'w', encoding='utf-8') as f:
        json.dump(summary, f, indent=2)

    print('Cascade protocol:', summary['protocol'])
    print(f'Patients={len(df)} | saved {out_csv}')
    print(f'Summary: {out_json}')
    if 'patient_binary_f1' in summary:
        print(
            f"Patient-level binary F1={summary['patient_binary_f1']:.4f} "
            f"Acc={summary['patient_binary_acc']:.4f}"
        )
    return {'summary': summary, 'df': df}


cascade_result = None
if RUN_CASCADE:
    cascade_result = run_full_volume_cascade(
        patient_records,
        max_patients=CASCADE_MAX_PATIENTS,
    )
else:
    print('SKIP cascade: ILD_RUN_CASCADE=0')


In [ ]:
# Evaluate cascade patient-level binary F1 across all saved fold checkpoints
import glob

ckpt_files = sorted(glob.glob(os.path.join(MODELS_DIR, 'hierarchical_fold_*.pth')))
all_f1s = []
all_rows = []

for ckpt in ckpt_files:
    res = run_full_volume_cascade(patient_records, ckpt_path=ckpt)
    if not res:
        continue
    f1 = float(res['summary'].get('patient_binary_f1', 0.0))
    all_f1s.append(f1)
    all_rows.append({
        'checkpoint': ckpt,
        'patient_binary_f1': f1,
        'patient_binary_acc': float(res['summary'].get('patient_binary_acc', 0.0)),
    })

fold_df = pd.DataFrame(all_rows)
if not fold_df.empty:
    display(fold_df)

if all_f1s:
    print(f'Across {len(all_f1s)} folds: F1 = {np.mean(all_f1s):.4f} +/- {np.std(all_f1s):.4f}')
else:
    print('No hierarchical_fold_*.pth checkpoints found or no cascade results were produced.')

## 6 — Bootstrap Confidence Intervals

Computes 95% percentile bootstrap CIs on all metrics by resampling folds with replacement.

In [ ]:
def bootstrap_metrics(all_results, metric_keys, n_boot=N_BOOT, seed=GLOBAL_SEED):
    """Compute bootstrap 95% CI for given metrics across all folds."""
    rng = np.random.RandomState(seed)
    n = len(all_results)
    boot_results = {key: [] for key in metric_keys}
    
    for _ in range(n_boot):
        idx = rng.choice(n, size=n, replace=True)
        for key in metric_keys:
            vals = []
            for i in idx:
                # Navigate nested dict
                parts = key.split('.')
                d = all_results[i]
                try:
                    for p in parts:
                        d = d[p]
                    if isinstance(d, (int, float)) and not isinstance(d, bool):
                        vals.append(float(d))
                except (KeyError, TypeError):
                    pass
            boot_results[key].append(np.mean(vals) if vals else 0.0)
    
    ci = {}
    for key, vals in boot_results.items():
        mean = float(np.mean(vals))
        lo = float(np.percentile(vals, 2.5))
        hi = float(np.percentile(vals, 97.5))
        ci[key] = {'mean': mean, 'ci_95': [lo, hi], 'std': float(np.std(vals))}
    return ci

metric_keys = [
    'binary.f1', 'binary.auc_roc', 'binary.accuracy', 'binary.precision',
    'binary.recall', 'binary.mcc', 'binary.kappa',
    'hierarchical.f1_macro', 'hierarchical.f1_weighted', 'hierarchical.auc_macro',
    'pathology.f1_macro', 'pathology.f1_weighted',
]
bootstrap_ci = bootstrap_metrics(all_results, metric_keys, n_boot=N_BOOT)

print('\n' + '=' * 60)
print('BOOTSTRAP 95% CONFIDENCE INTERVALS')
print('=' * 60)
print(f'{N_BOOT} bootstrap iterations')
for key in metric_keys:
    if key in bootstrap_ci:
        r = bootstrap_ci[key]
        print(f'  {key:30s}: {r["mean"]:.4f}  (95% CI [{r["ci_95"][0]:.4f}, {r["ci_95"][1]:.4f}])')

with open(os.path.join(EXPORTS_DIR, 'hierarchical_bootstrap_metrics.json'), 'w') as f:
    json.dump(bootstrap_ci, f, indent=2)
print(f'Saved: {EXPORTS_DIR}/hierarchical_bootstrap_metrics.json')

## 7 — ROC Curves & AUC (Binary and Multi-Class)

In [ ]:
# Load all validation probabilities from saved .npz files
all_y_bin, all_prob_bin = [], []
all_y_hier, all_prob_hier = [], []

for repeat in range(N_REPEATS):
    for fold in range(N_FOLDS):
        tag = f'R{repeat}F{fold}'
        path = os.path.join(EXPORTS_DIR, f'hier_val_probs_{tag}.npz')
        if os.path.isfile(path):
            d = np.load(path)
            all_y_bin.append(d['y_bin'])
            all_prob_bin.append(d['probs_bin'])
            all_y_hier.append(d['y_hier'])
            all_prob_hier.append(d['probs_hier'])

y_bin_all = np.concatenate(all_y_bin) if all_y_bin else np.array([])
prob_bin_all = np.concatenate(all_prob_bin) if all_prob_bin else np.array([])
y_hier_all = np.concatenate(all_y_hier) if all_y_hier else np.array([])
prob_hier_all = np.concatenate(all_prob_hier) if all_prob_hier else np.array([])

print(f'Concatenated: binary n={len(y_bin_all)}, hierarchical n={len(y_hier_all)}')

# ── Binary ROC ──
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Binary ROC
if len(y_bin_all) > 0 and len(np.unique(y_bin_all)) > 1:
    fpr, tpr, _ = roc_curve(y_bin_all, prob_bin_all[:, 1])
    auc = roc_auc_score(y_bin_all, prob_bin_all[:, 1])
    axes[0].plot(fpr, tpr, 'b-', linewidth=2, label=f'Binary ILD (AUC={auc:.3f})')
    axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title('Binary: Normal vs Any ILD')
    axes[0].legend(loc='lower right')
    axes[0].grid(True, alpha=0.3)

# Multi-class ROC (one-vs-rest)
if len(y_hier_all) > 0:
    rng = np.random.RandomState(GLOBAL_SEED)
    macro_aucs = []
    for i, cls_name in enumerate(HIERARCHY_CLASSES):
        y_bin = (y_hier_all == i).astype(int)
        if y_bin.sum() == 0 or y_bin.sum() == len(y_bin):
            continue
        auc = roc_auc_score(y_bin, prob_hier_all[:, i])
        fpr, tpr, _ = roc_curve(y_bin, prob_hier_all[:, i])
        boot_aucs = []
        for _ in range(N_BOOT):
            idx = rng.choice(len(y_bin), len(y_bin), replace=True)
            if len(np.unique(y_bin[idx])) < 2: continue
            boot_aucs.append(roc_auc_score(y_bin[idx], prob_hier_all[idx, i]))
        ci = [np.percentile(boot_aucs, 2.5), np.percentile(boot_aucs, 97.5)] if boot_aucs else [auc, auc]
        macro_aucs.append(auc)
        axes[1].plot(fpr, tpr, label=f'{cls_name} (AUC={auc:.3f} CI=[{ci[0]:.3f},{ci[1]:.3f}])')
    axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.5)
    axes[1].set_xlabel('False Positive Rate')
    axes[1].set_ylabel('True Positive Rate')
    axes[1].set_title(f'3-Class Hierarchical (macro-AUC={np.mean(macro_aucs):.3f})')
    axes[1].legend(loc='lower right', fontsize=8)
    axes[1].grid(True, alpha=0.3)

plt.tight_layout()
for d in (FIGURES_DIR, os.path.join('Results', 'figures')):
    os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, 'roc_curves_hierarchical.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: roc_curves_hierarchical.png')

## 8 — Calibration Analysis (ECE, Brier, Reliability)

In [ ]:
def compute_calibration(y_true, probs, n_bins=10):
    """Compute ECE and Brier score from real probabilities."""
    conf = probs.max(axis=1)
    pred = probs.argmax(axis=1)
    correct = (pred == y_true).astype(float)

    onehot = np.eye(probs.shape[1])[y_true]
    brier = float(np.mean(np.sum((probs - onehot) ** 2, axis=1)))

    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    bin_data = []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            bc, ba = float(conf[m].mean()), float(correct[m].mean())
            ece += (m.sum() / len(conf)) * abs(ba - bc)
            bin_data.append({
                'bin_center': float((bins[i] + bins[i + 1]) / 2),
                'confidence': bc,
                'accuracy': ba,
                'count': int(m.sum()),
            })
    return float(ece), brier, bin_data


def bootstrap_calibration_ci(y_true, probs, n_boot=N_BOOT, n_bins=10):
    rng = np.random.RandomState(GLOBAL_SEED)
    ece_boot, brier_boot = [], []
    n = len(y_true)
    for _ in range(n_boot):
        idx = rng.choice(n, n, replace=True)
        e, b, _ = compute_calibration(y_true[idx], probs[idx], n_bins)
        ece_boot.append(e)
        brier_boot.append(b)
    return {
        'ece_mean': float(np.mean(ece_boot)),
        'ece_ci_95': [float(np.percentile(ece_boot, 2.5)), float(np.percentile(ece_boot, 97.5))],
        'brier_mean': float(np.mean(brier_boot)),
        'brier_ci_95': [float(np.percentile(brier_boot, 2.5)), float(np.percentile(brier_boot, 97.5))],
        'n_samples': int(n),
        'n_boot': int(n_boot),
    }


calibration_out = {}

if len(y_bin_all) > 0 and len(prob_bin_all) > 0:
    ece_b, brier_b, bins_b = compute_calibration(y_bin_all, prob_bin_all)
    cal_bin = bootstrap_calibration_ci(y_bin_all, prob_bin_all)
    cal_bin['ece_point'] = ece_b
    cal_bin['brier_point'] = brier_b
    cal_bin['bins'] = bins_b
    calibration_out['binary'] = cal_bin
    print(
        f"Binary ECE={cal_bin['ece_mean']:.4f} "
        f"[{cal_bin['ece_ci_95'][0]:.4f}, {cal_bin['ece_ci_95'][1]:.4f}] | "
        f"Brier={cal_bin['brier_mean']:.4f} "
        f"[{cal_bin['brier_ci_95'][0]:.4f}, {cal_bin['brier_ci_95'][1]:.4f}]"
    )
else:
    cal_bin = None
    print('SKIP binary calibration: no concatenated probabilities')

if len(y_hier_all) > 0 and len(prob_hier_all) > 0:
    ece_h, brier_h, bins_h = compute_calibration(y_hier_all, prob_hier_all)
    cal_hier = bootstrap_calibration_ci(y_hier_all, prob_hier_all)
    cal_hier['ece_point'] = ece_h
    cal_hier['brier_point'] = brier_h
    cal_hier['bins'] = bins_h
    calibration_out['hierarchical'] = cal_hier
    print(
        f"Hier ECE={cal_hier['ece_mean']:.4f} "
        f"[{cal_hier['ece_ci_95'][0]:.4f}, {cal_hier['ece_ci_95'][1]:.4f}] | "
        f"Brier={cal_hier['brier_mean']:.4f} "
        f"[{cal_hier['brier_ci_95'][0]:.4f}, {cal_hier['brier_ci_95'][1]:.4f}]"
    )
else:
    cal_hier = None
    print('SKIP hierarchical calibration: no concatenated probabilities')

with open(os.path.join(EXPORTS_DIR, 'hierarchical_calibration.json'), 'w') as f:
    json.dump(calibration_out, f, indent=2)
print(f'Saved: {EXPORTS_DIR}/hierarchical_calibration.json')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, cal, title in [
    (axes[0], cal_bin, 'Binary Normal vs ILD'),
    (axes[1], cal_hier, '3-Class Hierarchical'),
]:
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.5, label='Perfect')
    if cal and cal.get('bins'):
        xs = [b['confidence'] for b in cal['bins']]
        ys = [b['accuracy'] for b in cal['bins']]
        cs = [b['count'] for b in cal['bins']]
        ax.plot(xs, ys, 'o-', label='Model')
        for x, y, c in zip(xs, ys, cs):
            ax.annotate(str(c), (x, y), textcoords='offset points', xytext=(4, 4), fontsize=7)
        ax.set_title(f"{title}\nECE={cal['ece_mean']:.3f}")
    else:
        ax.set_title(f'{title} (no data)')
    ax.set_xlabel('Confidence')
    ax.set_ylabel('Accuracy')
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
for d in (FIGURES_DIR, os.path.join('Results', 'figures')):
    os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, 'reliability_hierarchical.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Saved: reliability_hierarchical.png')


## 9 — Confusion Matrices & ROC JSON


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
cm_payload = {}

if len(y_bin_all) > 0:
    y_pred_bin = prob_bin_all.argmax(axis=1)
    cm_bin = confusion_matrix(y_bin_all, y_pred_bin, labels=[0, 1])
    cm_payload['binary'] = {'matrix': cm_bin.tolist(), 'classes': BINARY_CLASSES}
    im = axes[0].imshow(cm_bin, cmap='Blues')
    axes[0].set_xticks([0, 1])
    axes[0].set_yticks([0, 1])
    axes[0].set_xticklabels(BINARY_CLASSES)
    axes[0].set_yticklabels(BINARY_CLASSES)
    axes[0].set_xlabel('Predicted')
    axes[0].set_ylabel('True')
    axes[0].set_title('Binary')
    for i in range(2):
        for j in range(2):
            axes[0].text(
                j, i, int(cm_bin[i, j]), ha='center', va='center',
                color='white' if cm_bin[i, j] > cm_bin.max() / 2 else 'black',
            )
    fig.colorbar(im, ax=axes[0], fraction=0.046)
else:
    axes[0].set_title('Binary (no data)')

if len(y_hier_all) > 0:
    y_pred_hier = prob_hier_all.argmax(axis=1)
    labels = list(range(N_HIER_CLASSES))
    cm_hier = confusion_matrix(y_hier_all, y_pred_hier, labels=labels)
    cm_payload['hierarchical'] = {'matrix': cm_hier.tolist(), 'classes': HIERARCHY_CLASSES}
    im = axes[1].imshow(cm_hier, cmap='Blues')
    axes[1].set_xticks(labels)
    axes[1].set_yticks(labels)
    axes[1].set_xticklabels(HIERARCHY_CLASSES, rotation=30, ha='right')
    axes[1].set_yticklabels(HIERARCHY_CLASSES)
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('True')
    axes[1].set_title('Hierarchical')
    for i in range(len(labels)):
        for j in range(len(labels)):
            axes[1].text(
                j, i, int(cm_hier[i, j]), ha='center', va='center',
                color='white' if cm_hier[i, j] > cm_hier.max() / 2 else 'black',
            )
    fig.colorbar(im, ax=axes[1], fraction=0.046)
else:
    axes[1].set_title('Hierarchical (no data)')

plt.tight_layout()
for d in (FIGURES_DIR, os.path.join('Results', 'figures')):
    os.makedirs(d, exist_ok=True)
    fig.savefig(os.path.join(d, 'confusion_hierarchical.png'), dpi=150, bbox_inches='tight')
plt.show()

with open(os.path.join(EXPORTS_DIR, 'hierarchical_confusion.json'), 'w') as f:
    json.dump(cm_payload, f, indent=2)
print('Saved: confusion_hierarchical.png / hierarchical_confusion.json')

roc_payload = {}
if len(y_bin_all) > 0 and len(np.unique(y_bin_all)) > 1:
    auc = float(roc_auc_score(y_bin_all, prob_bin_all[:, 1]))
    rng = np.random.RandomState(GLOBAL_SEED)
    boots = []
    for _ in range(N_BOOT):
        idx = rng.choice(len(y_bin_all), len(y_bin_all), replace=True)
        if len(np.unique(y_bin_all[idx])) < 2:
            continue
        boots.append(roc_auc_score(y_bin_all[idx], prob_bin_all[idx, 1]))
    roc_payload['binary_auc'] = {
        'mean': auc,
        'ci_95': [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if boots else [auc, auc],
    }

if len(y_hier_all) > 0:
    per_class = {}
    macro = []
    rng = np.random.RandomState(GLOBAL_SEED)
    for i, name in enumerate(HIERARCHY_CLASSES):
        yb = (y_hier_all == i).astype(int)
        if yb.sum() == 0 or yb.sum() == len(yb):
            continue
        auc = float(roc_auc_score(yb, prob_hier_all[:, i]))
        boots = []
        for _ in range(N_BOOT):
            idx = rng.choice(len(yb), len(yb), replace=True)
            if len(np.unique(yb[idx])) < 2:
                continue
            boots.append(roc_auc_score(yb[idx], prob_hier_all[idx, i]))
        per_class[name] = {
            'auc': auc,
            'ci_95': [float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5))] if boots else [auc, auc],
        }
        macro.append(auc)
    roc_payload['hierarchical'] = {
        'per_class': per_class,
        'macro_auc': float(np.mean(macro)) if macro else None,
    }

with open(os.path.join(EXPORTS_DIR, 'hierarchical_roc.json'), 'w') as f:
    json.dump(roc_payload, f, indent=2)
print(f'Saved: {EXPORTS_DIR}/hierarchical_roc.json')


## 10 — Ablation Config Log (retrain = NB06)

Heavy retrain ablations are in `03_hierarchical_ablation.ipynb`. This cell only logs main-run settings.


In [ ]:
# Config log only — heavy retrain ablations live in 03_hierarchical_ablation.ipynb
RUN_ABLATIONS = os.environ.get('ILD_RUN_ABLATIONS', '0') == '1'
ablation_results = {
    'note': (
        'Retrain ablations are implemented in Experimentations/03_hierarchical_ablation.ipynb. '
        'This cell only logs the main CV config / summaries for traceability.'
    ),
    'notebook': '03_hierarchical_ablation.ipynb',
    'full_model': cv_results.get('binary_summary', {}) if 'cv_results' in dir() else {},
    'hierarchical': cv_results.get('hier_summary', {}) if 'cv_results' in dir() else {},
    'pathology': cv_results.get('path_summary', {}) if 'cv_results' in dir() else {},
    'factors': {
        'mixup': MIXUP_ALPHA,
        'label_smoothing': LABEL_SMOOTHING,
        'unfreeze_blocks': list(UNFREEZE_BLOCKS),
        'n_repeats': N_REPEATS,
        'n_folds': N_FOLDS,
    },
}
out = os.path.join(EXPORTS_DIR, 'hierarchical_ablation_config_log.json')
with open(out, 'w') as f:
    json.dump(ablation_results, f, indent=2)
print('NB01 ablation: config log only ->', out)
print('Run 03_hierarchical_ablation.ipynb for controlled retrain factors (binary F1).')
if RUN_ABLATIONS:
    print('NOTE: ILD_RUN_ABLATIONS=1 is ignored here; use NB06.')


## 11 — Per-patient Volumetric Biomarkers


In [ ]:
# Volumetric biomarkers: prefer cascade 3D pathology maps (proposal);
# fall back to ROI/seg labels only if maps are missing.

rows = []
cascade_dir = os.path.join(EXPORTS_DIR, 'cascade_maps')
vc = VolumeCache(MEDGIFT_ROOT, max_patients=1)

for rec in patient_records:
    try:
        map_path = os.path.join(cascade_dir, f"map_{rec['pid']}.npz")
        used_cascade = False
        if os.path.isfile(map_path):
            d = np.load(map_path)
            seg = d['pathology_map']
            lung = d['lung_mask'] if 'lung_mask' in d.files else None
            used_cascade = True
        else:
            packed = vc.get(rec['pid'], rec['path'])
            if packed is None:
                continue
            seg = packed.get('seg_labels')
            lung = packed.get('lung_mask')
            if seg is None:
                continue

        lung_vox = int((lung > 0.5).sum()) if lung is not None else int((seg >= 0).sum())
        counts = {name: int((seg == c).sum()) for c, name in enumerate(ORIGINAL_CLASS_NAMES)}
        path_vox = sum(counts[n] for n in ORIGINAL_CLASS_NAMES[1:])
        fibrotic = counts['Fibrosis'] + counts['Consolidation']
        nonfib = counts['Emphysema'] + counts['Ground Glass'] + counts['Micronodules']
        denom = max(lung_vox, 1)
        rows.append({
            'pid': rec['pid'],
            'group': rec['group'],
            'cohort': rec.get('cohort'),
            'source': 'cascade_map' if used_cascade else 'roi_gt',
            'has_ild': int(rec.get('has_ild', path_vox > 0)),
            'lung_voxels': lung_vox,
            'pathology_voxels': path_vox,
            'pathology_frac': path_vox / denom,
            'fibrotic_frac': fibrotic / denom,
            'nonfibrotic_frac': nonfib / denom,
            **{f'frac_{k.replace(" ", "_")}': counts[k] / denom for k in ORIGINAL_CLASS_NAMES[1:]},
        })
    except Exception as e:
        print('biomarker skip', rec.get('pid'), type(e).__name__, e)

biomarker_df = pd.DataFrame(rows)
out_csv = os.path.join(EXPORTS_DIR, 'hierarchical_biomarker.csv')
biomarker_df.to_csv(out_csv, index=False)
n_cas = (
    int((biomarker_df['source'] == 'cascade_map').sum())
    if len(biomarker_df) and 'source' in biomarker_df.columns else 0
)
print(f'Saved {len(biomarker_df)} patients -> {out_csv} (cascade_maps={n_cas})')
if len(biomarker_df):
    print(biomarker_df.describe().T[['mean', 'std', 'min', 'max']].round(4))
